# 02. 통계 비교와 LOT 선정

**담당:** 정수진 · **Issue:** #65  
**대상:** `P1-CR1-CAL01` 소둔로 · `P1-CR1-WLD01` 용접 기록  
**분석 단위:** LOT·공정·시간 구간·센서별 관측  
**재구성일:** 2026-10-11

## 분석 목적

**분석 목적:** 관측 품질과 운전 형태를 분리하고 같은 강종·같은 상태·같은 경과시간의 차이를 정량화한다. 공정별 전체 파형을 보존하며 대조 가능 시간, 지속 시간, 실제 온도 차이와 동시 신호를 근거로 검증할 LOT·센서·시간창을 좁힌다.

네 파일의 연결: **01 시간 흐름 → 02 통계와 LOT 선정 → 03 가설 검증 → 04 모델링 준비**. 각 파일은 같은 8개 원본 CSV를 독립적으로 읽어 실행합니다. 분석 표는 노트북 출력에 남기며, 그림만 지정된 `figures/05_all_data_timeseries_jinny_02`에 저장합니다.

## 02 상세 목차 — 무엇을 보고 무엇을 확인할 것인가

| 순서 | 분석 항목 | 볼 자료와 핵심 질문 | 확인 결과와 연결 |
|---:|---|---|---|
| 1 | 관측 품질 | LOT 누락 원인 → 센서 추가 결측 → 전체·초기 유효율 → TC/OP 원본: 무엇이 비어 있고 어느 비교에서 제외할 것인가? | OP-Z3 2 LOT 초기 결측과 공통 시각 누락 구분; 센서·분 품질 마스크 |
| 2 | 상태와 비교 조건 | 초기 승온·종료 전환·LOT별 관측 분·실제 작업 겹침 분: 비교 조건이 같은가? | 관측 범위·작업별 시간·합집합·중복을 분리 |
| 3 | 강종별 전체 공정 | 세정 → 건조 → 가열·균열 → 과시효 → 냉각 TC/OP/CP: 어디서 차이가 생기는가? | 전체 시간 보존, 색·번호·검토 이유·함께 볼 자료·실제 시각·다음 검증 연결 |
| 4 | 수준·변동·추세 | 상태별 중앙값·IQR·초반/후반 차이·기울기: 어떤 종류의 변화인가? | 실제 단위의 효과 크기와 상태별 표 |
| 5 | 동일 조건 대조 | 같은 강종·상태·경과 분의 차이와 대조 LOT 수: 어디까지 비교 가능한가? | 대조 부족 구간·지속 후보 시간창 |
| 6 | 연관성 단서 | TC↔해당 OP의 차분 상관·시차·CP 동시 변화: 공통 추세가 만든 상관인가? | 조건부 연관 단서, 인과·고장 판정 보류 |
| 7 | 후보와 문턱 민감도 | 온도 차이·지속 시간·대조 수·CP 원본 피크: 선정이 문턱에 민감한가? | 전수 후보·반대 사례·비교 불가 LOT |
| 8 | 03 전달 | LOT·공정·시간창·선정 이유·검증 질문: 무엇을 검증할 것인가? | 강종별 대조와 가설 검증 대상 |

## 기본 설정

관측 시작·종료는 제공된 기록의 범위이며 소재의 실제 입·출측 시각이 아닙니다. TC의 ℃와 OP의 %는 기존 분석의 표시 가정을 유지합니다. 원본에 단위 사전·목표값·운전 모드가 없어 공학적 정상 한계는 정하지 않습니다. CP-Z4M은 정의와 단위가 확인되지 않아 원본 값을 그대로 표시합니다.

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")
FIG_DIR = os.path.join("..", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")

ENC = "utf-8-sig"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

## 자료 읽기와 연결

LOT 시작과 CR1 강종변경 시각의 정확 일치를 확인하고, CAL01 설비의 WO와 부품교체를 실제 키로 연결합니다. 온도 LOT과 용접 COIL 사이의 공통 소재 키는 없으므로 시간 겹침은 동시 관측의 근거로만 사용합니다. 수리의 마지막 날짜는 하루 전체를 포함하는 계획으로 읽습니다.

In [ ]:
NOTEBOOK_NO = "02"
from pathlib import Path
from IPython.display import display, Markdown, Image
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

WLD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
files = {"온도": TEMP_CSV, "조업": EVENT_CSV, "수리": REPAIR_CSV,
         "설비": EQP_CSV, "부품": PART_CSV, "정비": MAINT_CSV,
         "교체": PARTCHG_CSV, "용접": WLD_CSV}
frames = {name: pd.read_csv(path, encoding=ENC) for name, path in files.items()}
temp, event, repair, eqp = [frames[k] for k in ["온도", "조업", "수리", "설비"]]
part, maint, partchg, wld = [frames[k] for k in ["부품", "정비", "교체", "용접"]]
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (repair, ["STRT_DT", "END_DT"]),
                       (maint, ["DETC_DT", "STRT_DT", "END_DT"]), (wld, ["MEAS_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])
CAL_TAG, WLD_TAG = "P1-CR1-CAL01", "P1-CR1-WLD01"
YEAR_START, YEAR_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
grade_evt = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT")
assert not grade_evt.EVT_DT.duplicated().any()
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_ROWS=("MEAS_DT", "size"))
lots = lots.merge(grade_evt[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
                  how="left", validate="one_to_one").rename(columns={"AFT_VAL": "GRADE"})
lots = lots.sort_values("START").reset_index(drop=True)
assert lots.GRADE.notna().all()
lots["MINUTES"] = (lots.END - lots.START).dt.total_seconds() / 60
lot_meta = lots.set_index("LOT_NO")
cal_work = maint.loc[maint.EQP_TAG.eq(CAL_TAG)].sort_values("STRT_DT").copy()
assert cal_work.WO_NO.is_unique and cal_work.END_DT.ge(cal_work.STRT_DT).all()
cal_parts = (partchg.merge(cal_work[["WO_NO", "EQP_TAG", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "REMARK"]],
                           on="WO_NO", validate="many_to_one")
             .merge(part, on="PART_CD", validate="many_to_one").sort_values("END_DT"))
plans = repair.loc[repair.PLANT_CD.eq("CR1")].sort_values("STRT_DT").copy()
plans["END_EXCL"] = plans.END_DT.dt.normalize() + pd.Timedelta(days=1)
coils = wld.groupby("COIL_ID", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_SEQ=("SEQ_NO", "size"),
    PWR_MIN=("PWR", "min"), PWR_MAX=("PWR", "max"), PWR_MEDIAN=("PWR", "median"))
coils["ZERO_ALL"] = coils.PWR_MIN.eq(0) & coils.PWR_MAX.eq(0)
coils = coils.sort_values("START")
TC = ["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
SIGNALS = TC + OP + ["CP-Z4", "CP-Z4M"]
HEAT = [f"TC-Z{i}" for i in range(3, 7)]
PROCESS = {"CLN-IN": "세정", **{f"TC-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 10)},
           "TC-OUT1": "냉각", "TC-OUT2": "냉각",
           **{f"OP-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 9)},
           "CP-Z4": "가열·균열", "CP-Z4M": "가열·균열"}
LOT_COLOR = {240001: "#455a64", 240024: "#d55e00", 240037: "#0072b2", 240038: "#0072b2",
             240044: "#0072b2", 240058: "#a23b57", 240061: "#e69f00", 240069: "#7b2cbf",
             240091: "#009e73", 240108: "#0072b2", 240114: "#e69f00", 240117: "#7b2cbf",
             240121: "#009e73", 240133: "#7b2cbf"}
COL = {"LOT": "#087f9c", "GRADE": "#477eb5", "WLD": "#de920a", "BM": "#b52a49",
       "CBM": "#d05c76", "TBM": "#e38592", "PART": "#7540a8", "PLAN": "#b8e2f2",
       "WORK": "#9a6324", "DETECT": "#34495e", "FOCUS": "#c83745", "QUALITY": "#bfc8cf"}
DUMMY_PRIOR = {"SPHC": "사용자 후보 가정", "SPCC": "사용자 후보 가정", "SPCE": "사용자 후보 가정",
               "SPCEN": "사용자 후보 가정", "SPFH590": "사용자 제외 가정", "DP590": "사용자 제외 가정",
               "SGCC": "사용자 제외 가정", "SPCD": "추가 검토"}
lots["DUMMY_PRIOR"] = lots.GRADE.map(DUMMY_PRIOR)
context_rows = []
for r in lots.itertuples():
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START, r.END)]
    wc = coils.loc[(coils.START <= r.END) & (coils.END >= r.START)]
    relationships = []
    for p in plans.itertuples():
        before = lots.loc[lots.END < p.STRT_DT].tail(1)
        after = lots.loc[lots.START >= p.END_EXCL].head(1)
        if len(before) and before.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 전 마지막 관측")
        if len(after) and after.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 후 첫 관측")
    context_rows.append({"LOT": int(r.LOT_NO), "GRADE": r.GRADE, "START": r.START, "END": r.END,
                         "MINUTES": r.MINUTES, "CAL_WORK": len(tasks), "CAL_DETECT": len(detected),
                         "WELD_COILS": len(wc), "PLAN_RELATION": " · ".join(relationships),
                         "DUMMY_PRIOR": DUMMY_PRIOR[r.GRADE]})
context = pd.DataFrame(context_rows).sort_values("START")
OUT_DIR = Path(FIG_DIR) / f"05_all_data_timeseries_jinny_{NOTEBOOK_NO}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
figure_checks = []
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

def note(text):
    display(Markdown(text))

def table(frame, columns=None):
    shown = frame if columns is None else frame[columns]
    # 작은 CP-Z4M도 0으로 반올림하지 않고 과학 표기법으로 보여준다.
    with pd.option_context("display.float_format", lambda value: f"{value:.4g}"):
        display(shown)


## 그림 표시 규칙

짧은 제목, 공정과 컬럼이 보이는 소제목, 축의 대상·단위, 색 범례를 사용합니다. TC는 왼쪽, 해당 OP는 오른쪽에 놓고 CP-Z4와 CP-Z4M도 왼쪽·오른쪽으로 대조합니다. TC-Z9에는 OP가 제공되지 않아 가상의 쌍을 만들지 않습니다. 모든 그림은 저장 크기에서 글자 겹침·잘림과 범례의 그래프 침범을 검사합니다.

In [ ]:
rcParams.update({"font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
                 "xtick.labelsize": 10, "ytick.labelsize": 10.5, "savefig.dpi": 160})

def unit(signal):
    if signal in TC:
        return "온도[℃]"
    if signal in OP:
        return "제어출력[%]"
    if signal == "CP-Z4M":
        return "CP-Z4M 기록값\n[정의·단위 미확인]"
    if signal == "CP-Z4":
        return "CP-Z4 기록값\n[단위 미확인]"
    return signal

def style(ax, xlabel="관측 시작 후 경과시간[분]"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#a9bac7")
    ax.grid(color="#e4ebf0", linewidth=.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.tick_params(pad=5)

def header(fig, title, subtitle="", handles=(), ncol=4):
    fig.suptitle(title, x=.035, y=.975, ha="left", va="top",
                 fontsize=20, fontweight="bold", color="#17344d")
    if subtitle:
        fig.text(.035, .924, subtitle, ha="left", va="top", fontsize=10.8, color="#526d80")
    if handles:
        fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.031, .879),
                   ncol=ncol, frameon=False, fontsize=10.5, handlelength=2.4,
                   columnspacing=1.8, labelspacing=.65)

def lot_handles(ids):
    return [Line2D([0], [0], color=LOT_COLOR[int(k)], lw=2,
                   label=f"{int(k)} · {lot_meta.loc[int(k), 'GRADE']} · {lot_meta.loc[int(k), 'START']:%m/%d}") for k in ids]

def text_box(artist, renderer):
    patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
    return patch.get_window_extent(renderer) if patch is not None else artist.get_window_extent(renderer)

def finish(fig, name, svg=False):
    """저장할 크기에서 모든 글자와 범례의 겹침·잘림을 확인한다."""
    fig.set_dpi(160)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    artists = list(fig.texts) + list(fig.legends)
    for ax in fig.axes:
        artists += list(ax.texts)
        if not ax.axison:
            continue
        artists += [ax.title, ax._left_title, ax.xaxis.label, ax.yaxis.label,
                    ax.xaxis.offsetText, ax.yaxis.offsetText] + list(ax.texts)
        for axis in [ax.xaxis, ax.yaxis]:
            low, high = sorted(axis.get_view_interval())
            for tick in axis.get_major_ticks():
                if low <= tick.get_loc() <= high:
                    artists += [tick.label1, tick.label2]
    unique = {}
    for artist in artists:
        if not artist.get_visible():
            continue
        if hasattr(artist, "get_text") and not artist.get_text():
            continue
        unique[id(artist)] = artist
    boxes = [(artist, text_box(artist, renderer)) for artist in unique.values()]
    for artist, box in boxes:
        if box.x0 < -.5 or box.y0 < -.5 or box.x1 > fig.bbox.x1 + .5 or box.y1 > fig.bbox.y1 + .5:
            raise RuntimeError(f"글자 잘림: {name}: {getattr(artist, 'get_text', lambda: '범례')()}")
    for i, (artist, box) in enumerate(boxes):
        for other, other_box in boxes[i + 1:]:
            if box.overlaps(other_box):
                raise RuntimeError(f"글자 겹침: {name}: {getattr(artist, 'get_text', lambda: '범례')()} / {getattr(other, 'get_text', lambda: '범례')()}")
    for legend in fig.legends:
        box = legend.get_window_extent(renderer)
        if any(box.overlaps(ax.get_window_extent(renderer)) for ax in fig.axes if ax.axison):
            raise RuntimeError(f"범례가 그래프를 가림: {name}")
    path = OUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, facecolor="white")
    if svg:
        vector = path.with_suffix(".svg")
        fig.savefig(vector, facecolor="white")
        vector.write_text("\n".join(line.rstrip() for line in vector.read_text(encoding="utf-8").splitlines()) + "\n", encoding="utf-8")
    figure_checks.append({"FILE": name, "TITLE": fig._suptitle.get_text(), "N_AXES": len(fig.axes),
                          "TEXT_OVERLAP": "통과", "TEXT_BOUNDS": "통과", "LEGEND_BOUNDS": "통과"})
    display(Image(filename=str(path), width=1440))
    plt.close(fig)


## 1. 관측 품질

**분석 목적:** 파형의 차이를 비교하기 전에, 값이 없는 이유와 시간을 확인한다. 아래 세 그림은 **① 전체 LOT의 누락 원인 → ② 추가 결측이 큰 두 LOT의 전체·초기 품질 → ③ 같은 시각 TC·OP 원본** 순서로 읽는다.

| 구분 | 제공 CSV에서 확인하는 것 | 분석에 반영할 것 |
|---|---|---|
| 시각 누락 | 기대 1초 시각에 행 자체가 없어 모든 컬럼이 같이 없음 | LOT·분당 표본 부족과 공통 누락 위치 |
| 센서 추가 NaN | 시각 행은 있지만 특정 센서값이 비어 있음 | 그 센서가 필요한 비교의 유효 구간 |
| 전체 유효율과 국소 결측 | 전체 비율이 높아도 일부 1분창은 값이 없을 수 있음 | 전체 LOT을 제외하기 전에 실제 결측 시간을 확인 |

1분은 LOT 관측 시작을 기준으로 나누며 마지막 분은 남은 초만 기대 표본으로 센다. **유효율 90%는 탐색용 비교 조건**이다. 정상·고장 한계나 통신 장애·센서 고장 원인을 판정하는 기준으로 사용하지 않는다.

### 1.1 누락 원인과 확인 대상


In [ ]:
# 1초 기대 간격은 원본 시각 차이의 최빈값으로 확인한다. 결측은 보간하지 않는다.
raw, profile, quality, expected, states = {}, {}, {}, {}, {}
quality_rows, state_rows = [], []
STATE_COLOR = {"안정 참고": "#009e73", "초기 승온": "#e69f00", "종료 전환": "#d55e00"}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = temp.loc[temp.LOT_NO.eq(k)].sort_values("MEAS_DT").copy()
    assert not g.MEAS_DT.duplicated().any()
    gaps_s = g.MEAS_DT.diff().dt.total_seconds().dropna()
    assert gaps_s.mode().iloc[0] == 1 and gaps_s.ge(1).all()
    g["ELAPSED"] = (g.MEAS_DT - r.START).dt.total_seconds() / 60
    g["MINUTE"] = np.floor(g.ELAPSED).astype(int)
    grid = pd.RangeIndex(int(np.floor(r.MINUTES)) + 1, name="MINUTE")
    p = g.groupby("MINUTE")[SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    exp = pd.Series(np.minimum(60, int((r.END-r.START).total_seconds()) + 1 - grid.to_numpy()*60), index=grid)
    counts = g.groupby("MINUTE").size().reindex(grid, fill_value=0)
    q = p.xs("count", axis=1, level=1).fillna(0).div(exp, axis=0)
    raw[k], profile[k], quality[k], expected[k] = g, p, q, exp
    for s in SIGNALS:
        quality_rows.append({"LOT": k, "GRADE": r.GRADE, "SENSOR": s, "ROWS": len(g),
                             "EXPECTED_SECONDS": int(exp.sum()), "MISSING_TIMES_S": int(exp.sum()-len(g)),
                             "SENSOR_NAN": int(g[s].isna().sum()), "VALID_RATIO": g[s].count()/exp.sum(),
                             "LOW_90_WINDOWS": int(q[s].lt(.9).sum())})
    # 사후 형태 분류다. 실제 운전 모드나 온라인 모델 입력으로 사용하지 않는다.
    st = pd.Series("안정 참고", index=grid, dtype="object")
    early = g.loc[g.ELAPSED.lt(5), HEAT].median()
    later = g.loc[g.ELAPSED.between(80,100), HEAT].median()
    rise = later - early
    warm_end, tail_start = np.nan, np.nan
    if rise.notna().all() and rise.min() >= 20 and g.ELAPSED.between(80,100).shape[0] >= 600:
        near = pd.concat([(p[(s,"median")]-later[s]).abs().le(5 if s=="TC-Z3" else 2) for s in HEAT], axis=1).all(axis=1)
        slow = pd.concat([p[(s,"median")].diff(5).abs().le(2) for s in HEAT[1:]], axis=1).all(axis=1)
        run = (near & slow & grid.to_series().ge(5)).rolling(5, min_periods=5).sum().eq(5)
        warm_end = int(run.index[run][0])+1 if run.any() else len(grid)
        st.loc[grid < warm_end] = "초기 승온"
    middle = g.loc[g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6), HEAT].median()
    tail = g.loc[g.ELAPSED.ge(max(0,r.MINUTES-5)), HEAT].median()
    if (tail-middle).notna().all() and (tail-middle).max() <= -10:
        midmask = g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6)
        drop = pd.concat([(p[(s,"median")]-middle[s]).le(-10) for s in HEAT], axis=1).sum(axis=1).ge(3)
        cp_drop = (p[("CP-Z4","median")]-g.loc[midmask,"CP-Z4"].median()).le(-.05).rolling(3).sum().eq(3)
        op = ["OP-Z4","OP-Z5","OP-Z6"]
        op_drop = (p.loc[:,[(s,"median") for s in op]].median(axis=1)-g.loc[midmask,op].median().median()).le(-10).rolling(3).sum().eq(3)
        hit = (drop | cp_drop | op_drop) & grid.to_series().ge(r.MINUTES*.6)
        if hit.any():
            tail_start = int(hit.index[hit][0])
            st.loc[grid >= tail_start] = "종료 전환"
    states[k] = st
    state_rows.append({"LOT": k, "GRADE": r.GRADE, "RISE_MIN_C": rise.min(), "RISE_MAX_C": rise.max(),
                       "WARM_END_MIN": warm_end, "TAIL_START_MIN": tail_start,
                       "STABLE_WINDOWS": int(st.eq("안정 참고").sum())})
quality_summary = pd.DataFrame(quality_rows)
state_summary = pd.DataFrame(state_rows)
def runs(mask, minimum=1):
    """연속 True의 [시작, 끝 제외) 경과 분을 돌려준다."""
    a = np.flatnonzero(mask.to_numpy(bool))
    if not len(a):
        return []
    groups = np.split(a, np.flatnonzero(np.diff(a)>1)+1)
    return [(int(mask.index[z[0]]), int(mask.index[z[-1]])+1) for z in groups if len(z)>=minimum]


In [ ]:
# 전체 누락을 공통 시각 누락과 센서별 추가 NaN으로 분리한다.
QCOL = {"TIME":"#62788a","NAN":"#d55e00","TC":"#0072b2","OP":"#009e73","CHECK":"#bd334a"}
lot_quality=quality_summary.groupby(["LOT","GRADE"],as_index=False).agg(
    EXPECTED_SECONDS=("EXPECTED_SECONDS","first"),ROWS=("ROWS","first"),
    MISSING_TIMES_S=("MISSING_TIMES_S","first"),TOTAL_SENSOR_NAN=("SENSOR_NAN","sum"),
    MIN_VALID_RATIO=("VALID_RATIO","min"),MAX_SENSOR_NAN=("SENSOR_NAN","max"))
lot_quality=lot_quality.set_index("LOT").reindex(lots.LOT_NO.to_numpy()).rename_axis("LOT").reset_index()
lot_quality["TIME_MISSING_PCT"]=lot_quality.MISSING_TIMES_S/lot_quality.EXPECTED_SECONDS*100
lot_quality["EXTRA_NAN_PCT"]=lot_quality.MAX_SENSOR_NAN/lot_quality.EXPECTED_SECONDS*100
lot_quality["WORST_MISSING_PCT"]=100-lot_quality.MIN_VALID_RATIO*100
assert np.allclose(lot_quality.WORST_MISSING_PCT,lot_quality.TIME_MISSING_PCT+lot_quality.EXTRA_NAN_PCT)
quality_summary["SENSOR_NAN_PCT"]=quality_summary.SENSOR_NAN/quality_summary.EXPECTED_SECONDS*100
nan_focus=quality_summary.nlargest(2,"SENSOR_NAN_PCT").copy()
remaining=quality_summary.drop(index=nan_focus.index)
other_max=remaining.loc[remaining.SENSOR_NAN_PCT.idxmax()]
focus_lots=nan_focus.LOT.astype(int).tolist()

note("**그림 목적 — 누락 원인 비교:** 모든 LOT에서 공통 시각 누락과 센서 추가 NaN을 나누고, 추가 결측이 집중된 LOT·컬럼을 찾는다. 왼쪽은 각 LOT의 최소 유효 센서, 오른쪽은 추가 결측률 상위 2개와 나머지 전체 조합의 최대값이다.")
fig,axs=plt.subplots(1,2,figsize=(25,12),gridspec_kw={"width_ratios":[1.25,1]})
fig.subplots_adjust(left=.16,right=.97,top=.77,bottom=.19,wspace=.68)
ax=axs[0];y=np.arange(len(lot_quality))
ax.barh(y,lot_quality.TIME_MISSING_PCT,color=QCOL["TIME"],height=.65)
ax.barh(y,lot_quality.EXTRA_NAN_PCT,left=lot_quality.TIME_MISSING_PCT,color=QCOL["NAN"],height=.65)
for i,r in enumerate(lot_quality.itertuples()):
    ax.text(r.WORST_MISSING_PCT+.09,i,f"{r.WORST_MISSING_PCT:.2f}%",va="center",fontsize=11.5)
    if r.LOT in focus_lots:ax.axhspan(i-.43,i+.43,color=QCOL["NAN"],alpha=.07,zorder=0,lw=0)
ax.set_yticks(y,[f"{r.LOT} · {r.GRADE}" for r in lot_quality.itertuples()]);ax.invert_yaxis()
for tick,r in zip(ax.get_yticklabels(),lot_quality.itertuples()):
    if r.LOT in focus_lots:tick.set_color(QCOL["NAN"]);tick.set_fontweight("bold")
ax.set_xlim(0,lot_quality.WORST_MISSING_PCT.max()+1.0)
ax.set_title("전체 LOT [유효율이 가장 낮은 센서]",loc="left",pad=13)
ax.set_ylabel("LOT · 강종[관측 시작순]",labelpad=10);style(ax,"기대 1초 표본 대비 누락률[%]")

ax=axs[1]
detail=nan_focus[["LOT","SENSOR","SENSOR_NAN_PCT","SENSOR_NAN","EXPECTED_SECONDS"]].copy()
values=detail.SENSOR_NAN_PCT.tolist()+[other_max.SENSOR_NAN_PCT]
labels=[f"{int(r.LOT)} · {r.SENSOR}" for r in detail.itertuples()]+[f"나머지 {len(remaining)} LOT·센서\n조합 중 최대"]
ax.barh(range(3),values,color=QCOL["NAN"],height=.45)
for i,(value,count) in enumerate(zip(values,detail.SENSOR_NAN.tolist()+[other_max.SENSOR_NAN])):
    ax.text(value+.045,i,f"{value:.2f}% · {int(count)}개",va="center",fontsize=12)
ax.set_yticks(range(3),labels);ax.set_ylim(2.6,-.7);ax.set_xlim(0,max(values)*1.55)
ax.set_title("추가 NaN [LOT · 센서 컬럼]",loc="left",pad=13)
ax.set_ylabel("확인 대상",labelpad=10);style(ax,"기대 1초 표본 대비 센서 NaN[%]")
header(fig,"누락 원인 비교","대부분은 모든 센서에 공통인 시각 누락; 240044·240061의 OP-Z3에 추가 NaN 집중",
       [Patch(facecolor=QCOL["TIME"],label="시각 누락 · 해당 시각의 모든 센서"),
        Patch(facecolor=QCOL["NAN"],label="추가 NaN · 시각은 있지만 센서값 없음")],ncol=2)
fig.text(.16,.065,f"봐야 할 대상: 두 LOT의 가열·균열 [OP-Z3]  |  나머지 {len(remaining)}개 조합의 추가 결측률은 {other_max.SENSOR_NAN_PCT:.2f}% 이하",
         fontsize=13,color="#17344d")
finish(fig,"01_품질상태/01_센서별_유효율.png")
note(f"**그림 결과:** LOT 공통 시각 누락은 {lot_quality.TIME_MISSING_PCT.min():.2f}–{lot_quality.TIME_MISSING_PCT.max():.2f}%다. 240044 OP-Z3는 추가 NaN 75개(1.67%), 240061 OP-Z3는 179개(1.32%)이며 나머지 {len(remaining)}개 조합은 최대 {other_max.SENSOR_NAN_PCT:.2f}%다. 반복되는 행 색을 읽는 대신 두 OP-Z3의 결측 시점을 다음 그림에서 확인한다.")
table(lot_quality)
table(nan_focus[["LOT","GRADE","SENSOR","SENSOR_NAN","SENSOR_NAN_PCT","VALID_RATIO","LOW_90_WINDOWS"]])
seq=wld.groupby("COIL_ID").SEQ_NO.agg(["count","nunique","min","max"])
assert seq["count"].eq(39).all() and seq["nunique"].eq(39).all()
note(f"**범위와 한계:** 시각 누락 {lot_quality.MISSING_TIMES_S.sum():,}초와 센서 NaN {lot_quality.TOTAL_SENSOR_NAN.sum():,}개는 서로 다른 단위이며 합산하지 않는다. 용접 {len(seq):,} COIL은 모두 39 SEQ다. 누락 원인 분류는 제공 CSV의 기록 상태를 구분한 결과이며 통신 장애·센서 고장을 확정하는 진단은 아니다.")


### 1.2 전체 관측에서 위치를 찾고 첫 5분 확대

**확인 질문:** 두 OP-Z3의 추가 결측은 관측 전체에 퍼져 있는가, 초기 특정 시간에 집중되는가? 전체 유효율과 분당 유효율을 비교하고, 기존 90% 비교 조건에서 제외되는 시간을 확인한다.

In [ ]:
# 전체 LOT 평균과 첫 5분의 분당 유효율을 같은 축 단위로 비교한다.
note("**그림 목적 — 전체·초기 유효율 비교:** OP-Z3의 전체 유효율이 높아도 초기 비교가 가능한지 확인한다. 상위 추가 결측 2 LOT을 대상으로 왼쪽 전체 관측에서 위치를 표시하고 오른쪽 첫 5분을 확대한다.")
fig,axs=plt.subplots(2,2,figsize=(24,12))
fig.subplots_adjust(left=.10,right=.97,top=.76,bottom=.18,wspace=.28,hspace=.70)
quality_review_rows=[]
for i,k in enumerate(focus_lots):
    r=lot_meta.loc[k];g=raw[k];q=quality[k];s="OP-Z3"
    missing=g.loc[g[s].isna(),"ELAPSED"]
    a=float(missing.min());b=float(missing.max()+1/60)
    whole=float(quality_summary.loc[quality_summary.LOT.eq(k)&quality_summary.SENSOR.eq(s),"VALID_RATIO"].iloc[0]*100)
    for j,ax in enumerate(axs[i]):
        end=r.MINUTES if j==0 else 5
        window=q.loc[q.index<=end]
        for sensor,key in [("TC-Z3","TC"),("OP-Z3","OP")]:
            x=np.r_[window.index.to_numpy(),min(end,window.index[-1]+1)]
            y=np.r_[window[sensor].to_numpy()*100,window[sensor].iloc[-1]*100]
            ax.step(x,y,where="post",color=QCOL[key],lw=2,ls="--" if key=="OP" else "-")
        ax.axhline(90,color=QCOL["CHECK"],ls="--",lw=1.2)
        ax.axvspan(a,min(b,end),color=QCOL["NAN"],alpha=.10,lw=0)
        ax.set_xlim(-.05,end);ax.set_ylim(-4,105)
        ax.set_yticks([0,25,50,75,100])
        ax.set_title(f"{k} · {r.GRADE} · {'전체 관측' if j==0 else '첫 5분 확대'}",loc="left",pad=12)
        ax.set_ylabel("기대 표본 대비 분당 유효율[%]",labelpad=10)
        style(ax,"관측 시작 후 경과시간[분]");ax.xaxis.set_major_locator(MaxNLocator(5))
        if j==1:
            ax.set_xticks(range(6))
            initial_zero=int(q[s].eq(0).cumprod().sum())
            initial_label="첫 1분 0%" if initial_zero==1 else f"초기 {initial_zero}개 1분창 0%"
            ax.text(2.6,14,f"OP-Z3 전체 {whole:.2f}%\n{initial_label}",fontsize=12,color=QCOL["OP"],
                    bbox=dict(boxstyle="round,pad=.3",fc="white",ec=QCOL["OP"],lw=.8))
    quality_review_rows.append({"LOT":k,"공정":"가열·균열","센서":"OP-Z3","NaN 개수":int(g[s].isna().sum()),
                                "NaN 최초 시각":g.loc[g[s].isna(),"MEAS_DT"].min(),
                                "NaN 마지막 시각":g.loc[g[s].isna(),"MEAS_DT"].max(),
                                "전체 유효율[%]":whole,"첫 1분 유효율[%]":q[s].iloc[0]*100,
                                "OP-Z3 90% 미만 1분창[개]":int(q[s].lt(.9).sum()),
                                "TC·OP 함께 유효한 1분창[개]":int(q[["TC-Z3","OP-Z3"]].ge(.9).all(axis=1).sum())})
quality_review=pd.DataFrame(quality_review_rows)
header(fig,"전체·초기 유효율 비교","전체 유효율 95–97%인 두 OP-Z3도 첫 1분은 0%; 전체 평균과 국소 결측을 나누어 확인",
       [Line2D([0],[0],color=QCOL["TC"],lw=2,label="가열·균열 [TC-Z3] 유효율"),
        Line2D([0],[0],color=QCOL["OP"],lw=2,ls="--",label="가열·균열 [OP-Z3] 유효율"),
        Line2D([0],[0],color=QCOL["CHECK"],ls="--",label="90% · 탐색용 비교 조건"),
        Patch(facecolor=QCOL["NAN"],alpha=.2,label="초기 센서 NaN 검토 위치")],ncol=4)
fig.text(.10,.065,"확인: 초기 TC 기록이 있어도 대응 OP가 없으면 TC↔OP 비교가 불가능한 분이 있다. 전체 LOT을 버리지 않고 해당 센서·분을 구분한다.",
         fontsize=12.5,color="#17344d")
finish(fig,"01_품질상태/01A_전체구간_초기결측.png")
note("**그림 결과:** 240044 OP-Z3 전체 유효율은 95.09%지만 첫 1분은 0%, 240061은 전체 97.01%지만 첫 3개 1분창이 0%다. 같은 시간 TC-Z3에는 유효한 기록이 있다. 90% 조건은 분석용 품질 문턱이며 정상·고장 기준이 아니다.")
table(quality_review)
note("**다음 확인과 분석 반영:** 초기 추가 결측의 실제 길이와 다른 컬럼의 존재 여부를 아래 원본에서 확인한다. TC·OP 연관성에는 두 센서가 모두 유효한 분만 사용하며, 나머지 TC 및 OP의 유효 구간은 보존한다. 짧은 결측을 포함한 분도 90%를 통과할 수 있으므로 원본 결측 표시를 함께 읽는다.")


### 1.3 같은 시각의 TC·OP 원본 대조

**확인 질문:** OP가 비어 있던 시간에도 TC가 존재하는가? 반대로 행 자체가 빠지면 두 센서는 함께 끊기는가? 첫 두 행은 추가 NaN 대상, 마지막 행은 공통 시각 누락의 원본 예시다. 공정 고장과 데이터 결측을 구분하기 위한 비교다.

In [ ]:
# 표시용 1초 격자에 놓아 누락 시각에서 선을 끊는다. 값을 보간하거나 0으로 채우지 않는다.
def second_intervals(mask):
    starts=mask & ~mask.shift(1,fill_value=False)
    ends=mask & ~mask.shift(-1,fill_value=False)
    return list(zip(mask.index[starts],mask.index[ends]+pd.Timedelta(seconds=1)))

second_quality={}
for k in focus_lots+[240069]:
    r=lot_meta.loc[k];g=raw[k].set_index("MEAS_DT")
    grid=pd.date_range(r.START,r.END,freq="s")
    present=pd.Series(grid.isin(g.index),index=grid)
    shown=g[SIGNALS].reindex(grid)
    absent=~present
    assert int(absent.sum())==int(quality_summary.loc[quality_summary.LOT.eq(k),"MISSING_TIMES_S"].iloc[0])
    for s in ["TC-Z3","OP-Z3"]:
        sensor_nan=present & shown[s].isna()
        assert int(sensor_nan.sum())==int(g[s].isna().sum())
        assert int(shown[s].count()+absent.sum()+sensor_nan.sum())==len(grid)
    second_quality[k]=(shown,present)

# 시각 누락의 예는 추가 NaN 대상과 별도로 고른다: 240069의 가장 긴 공통 누락.
shown,present=second_quality[240069]
gap=max(second_intervals(~present),key=lambda z:z[1]-z[0])
cases=[(k,lot_meta.loc[k,"START"],lot_meta.loc[k,"START"]+pd.Timedelta(minutes=5),"센서 추가 NaN · 첫 5분") for k in focus_lots]
cases.append((240069,gap[0]-pd.Timedelta(seconds=10),gap[1]+pd.Timedelta(seconds=10),"공통 시각 누락 · 최장 10초 주변"))
note("**그림 목적 — 시각 누락·센서 NaN 원본:** 왼쪽 TC-Z3와 오른쪽 OP-Z3를 같은 실제 시각에서 대조한다. 위 두 행은 OP 추가 결측, 마지막 행은 모든 컬럼에 같은 시각의 행이 없는 경우다. 두 종류의 빈 선을 구분한다.")
fig,axs=plt.subplots(3,2,figsize=(24,15))
fig.subplots_adjust(left=.10,right=.97,top=.77,bottom=.17,wspace=.30,hspace=.85)
for i,(k,a,b,reason) in enumerate(cases):
    shown,present=second_quality[k]
    values=shown.loc[a:b];has_row=present.loc[a:b]
    for j,(s,key) in enumerate([("TC-Z3","TC"),("OP-Z3","OP")]):
        ax=axs[i,j]
        for start,end in second_intervals(~has_row):
            ax.axvspan(start,end,color=QCOL["TIME"],alpha=.22,lw=0)
        for start,end in second_intervals(has_row & values[s].isna()):
            ax.axvspan(start,end,color=QCOL["NAN"],alpha=.20,lw=0)
        ax.plot(values.index,values[s],color=QCOL[key],lw=1.8)
        ax.set_xlim(a,b)
        ax.set_title(f"{k} · {lot_meta.loc[k,'GRADE']} · 가열·균열 [{s}] · {a:%m/%d}",loc="left",pad=12)
        ax.set_ylabel(unit(s),labelpad=10);style(ax,"실제 관측 시각[시:분:초]")
        ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=2) if i<2 else mdates.SecondLocator(interval=10))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.yaxis.set_major_locator(MaxNLocator(4))
        if s=="OP-Z3":
            ax.set_ylim(-3,103);ax.set_yticks([0,25,50,75,100])
        else:
            ax.set_ylim(np.floor(values[s].min())-1,np.ceil(values[s].max())+1)
        if i<2 and s=="OP-Z3":
            missing=values.index[has_row & values[s].isna()]
            duration=int((missing.max()-missing.min()).total_seconds()+1)
            midpoint=missing.min()+(missing.max()-missing.min())/2
            ax.text(midpoint,.50,f"OP값 없음 · {duration}초\n센서 NaN {len(missing)}개",
                    transform=ax.get_xaxis_transform(),ha="center",va="center",fontsize=11.5,color=QCOL["NAN"])
        elif i==2:
            midpoint=gap[0]+(gap[1]-gap[0])/2
            ax.text(midpoint,.50,"행 자체가 없음\n모든 센서 · 10초",
                    transform=ax.get_xaxis_transform(),ha="center",va="center",fontsize=11.5,color=QCOL["TIME"])
header(fig,"시각 누락·센서 NaN 대조","TC 왼쪽 · OP 오른쪽; 주황은 해당 센서만 NaN, 회색은 해당 시각의 행 누락; 빈 선은 0이 아님",
       [Line2D([0],[0],color=QCOL["TC"],lw=2,label="가열·균열 [TC-Z3] 원본"),
        Line2D([0],[0],color=QCOL["OP"],lw=2,label="가열·균열 [OP-Z3] 원본"),
        Patch(facecolor=QCOL["TIME"],alpha=.35,label="시각 누락 · 모든 센서 공통"),
        Patch(facecolor=QCOL["NAN"],alpha=.3,label="센서 추가 NaN · 다른 컬럼은 존재")],ncol=4)
fig.text(.10,.065,"확인: 두 LOT의 초기 OP 결측과 240069의 공통 10초 누락은 다른 기록 문제다. 온도·출력의 급락이나 설비 고장으로 바꾸어 해석하지 않는다.",
         fontsize=12.5,color="#17344d")
finish(fig,"01_품질상태/01B_시각누락_센서결측_원본.png")
for k in focus_lots:
    g=raw[k];missing=g.loc[g["OP-Z3"].isna(),"MEAS_DT"]
    envelope_s=(missing.max()-missing.min()).total_seconds()+1
    nan_s=int(len(missing))
    common_s=int((~second_quality[k][1].loc[missing.min():missing.max()]).sum())
    note(f"**그림 결과 — {k}:** OP-Z3의 초기 결측은 {missing.min():%m/%d %H:%M:%S}–{missing.max():%H:%M:%S}, 최초~마지막 NaN을 포함한 {envelope_s:.0f}초 범위에 있다. 이 범위에서 OP 센서 NaN {nan_s}개와 공통 시각 누락 {common_s}초를 구분한다. 같은 범위 TC-Z3에는 {int(second_quality[k][0].loc[missing.min():missing.max(),'TC-Z3'].count())}개 유효 값이 있다.")
note(f"**그림 결과 — 240069:** {gap[0]:%m/%d %H:%M:%S}–{gap[1]-pd.Timedelta(seconds=1):%H:%M:%S}의 {(gap[1]-gap[0]).total_seconds():.0f}초는 행 자체가 없다. 모든 22개 센서가 같은 시간에 빠져 왼쪽 TC와 오른쪽 OP가 함께 끊긴다. 통신·수집·센서 고장 중 어느 원인인지는 CSV만으로 확인되지 않는다.")
note("**분석 결정:** 두 OP-Z3 초기 결측은 센서·시간 범위를 표시하고 TC↔OP 비교 품질 조건에 반영한다. 공통 시각 누락은 전체 LOT별 품질 마스크로 계속 확인한다. 두 종류 모두 보간·0 채움 없이 원본으로 유지하며, 유효율이 높다는 이유로 정상 라벨을 붙이지 않는다.")


## 2. 상태와 비교 조건

**목적:** 초기 승온과 종료 전환의 관측 형태를 안정 참고와 분리한다. 4개 가열대의 초기 5분 대비 80–100분 차이와 마지막 5분 대비 중간 45–60% 차이를 사용하므로 사후 검토 상태다. 실제 작업·감지와 품질은 상태를 덮어쓰지 않는다. 동일 강종이라도 두께·속도·목표 온도 정보가 없다는 한계가 남는다.

**시간 정의:** 그림 오른쪽의 관측 범위는 최초–마지막 기록의 경과 분이다. 실제 CAL01 작업 겹침은 관측 범위와 STRT_DT–END_DT의 교집합이며 여러 작업의 겹침은 합집합으로 중복을 제외한다. 계획 수리 날짜·감지 시각과 실제 작업 겹침을 구분한다.

In [ ]:
# 관측 범위와 실제 CAL01 작업의 교집합을 계산한다. 같은 시각의 작업은 한 번만 센다.
def merge_ranges(intervals):
    merged=[]
    for start,end in sorted(intervals):
        if end<=start:
            continue
        if merged and start<=merged[-1][1]:
            merged[-1]=(merged[-1][0],max(end,merged[-1][1]))
        else:
            merged.append((start,end))
    return merged

work_overlap_rows=[]
work_ranges={}
time_rows=[]
for r in lots.itertuples():
    k=int(r.LOT_NO)
    tasks=cal_work.loc[cal_work.STRT_DT.lt(r.END)&cal_work.END_DT.gt(r.START)]
    intervals=[]
    for t in tasks.itertuples():
        start=max(r.START,t.STRT_DT);end=min(r.END,t.END_DT)
        a=(start-r.START).total_seconds()/60;b=(end-r.START).total_seconds()/60
        intervals.append((a,b))
        work_overlap_rows.append({"LOT":k,"WO_NO":t.WO_NO,"정비유형":t.MNT_TYPE,"REMARK":t.REMARK,
                                  "겹침 시작":start,"겹침 종료":end,"겹침 시작[분]":a,"겹침 종료[분]":b,
                                  "작업별 겹침[분]":b-a})
    work_ranges[k]=merge_ranges(intervals)
    union=sum(b-a for a,b in work_ranges[k])
    summed=sum(b-a for a,b in intervals)
    assert 0<=union<=r.MINUTES+1e-9 and union<=summed+1e-9
    time_rows.append({"LOT":k,"강종":r.GRADE,"관측 시작":r.START,"관측 종료":r.END,
                      "관측 범위[분]":r.MINUTES,"기록 행수[개]":r.N_ROWS,
                      "작업 겹침 합집합[분]":union,"작업별 겹침 단순합[분]":summed,
                      "중복 집계분[분]":summed-union})
observation_times=pd.DataFrame(time_rows)
work_overlaps=pd.DataFrame(work_overlap_rows)
time_meta=observation_times.set_index("LOT")
# 240117의 두 작업은 41분 겹친다. 관측과의 겹침은 115+230-41=304분이다.
assert np.isclose(time_meta.loc[240117,"작업 겹침 합집합[분]"],304)
assert np.isclose(time_meta.loc[240117,"중복 집계분[분]"],41)
assert observation_times["작업 겹침 합집합[분]"].gt(0).sum()==1

note("**그림 목적 — 관측 상태·작업 시간:** LOT별 관측 범위와 실제 CAL01 작업이 겹친 시간을 분으로 비교한다. 오른쪽 작업 시간은 STRT_DT–END_DT와 관측 범위의 교집합이며, 동시에 수행한 작업은 합집합으로 한 번만 센다.")
fig,axs=plt.subplots(1,2,figsize=(25,11),gridspec_kw={"width_ratios":[4.4,1]},sharey=True)
fig.subplots_adjust(left=.16,right=.98,top=.76,bottom=.20,wspace=.045)
ax=axs[0];info=axs[1]
for i,r in enumerate(lots.itertuples()):
    k=int(r.LOT_NO)
    for state,color in STATE_COLOR.items():
        for a,b in runs(states[k].eq(state)):
            end=min(b,r.MINUTES)
            if end>a:ax.broken_barh([(a,end-a)],(i-.29,.58),facecolors=color)
    for a,b in work_ranges[k]:
        ax.broken_barh([(a,b-a)],(i-.32,.64),facecolors="none",edgecolors=COL["WORK"],hatch="////",lw=1)
    info.text(.06,i,f'{r.MINUTES:.1f}',ha="left",va="center",fontsize=11.5,color="#17344d")
    value=time_meta.loc[k,"작업 겹침 합집합[분]"]
    info.text(.59,i,f'{value:.1f}',ha="left",va="center",fontsize=11.5,
              color=COL["WORK"] if value else "#7b8a95",fontweight="bold" if value else "normal")
ax.set_yticks(range(len(lots)),[f"{r.LOT_NO} · {r.GRADE} · {r.START:%m/%d}" for r in lots.itertuples()])
ax.set_ylim(len(lots)-.4,-.8);ax.set_xlim(0,lots.MINUTES.max()+15)
ax.set_ylabel("LOT · 강종 · 관측 시작일",labelpad=12);style(ax)
info.set_xlim(0,1);info.set_axis_off()
info.text(.06,1.02,"관측 범위\n[분]",transform=info.transAxes,ha="left",va="bottom",fontsize=11.5,color="#17344d")
info.text(.59,1.02,"작업 겹침\n[분]",transform=info.transAxes,ha="left",va="bottom",fontsize=11.5,color=COL["WORK"])
header(fig,"관측 상태·작업 시간 비교","오른쪽 관측 범위=최초–마지막 기록의 경과시간; 작업 겹침=실제 CAL01 작업의 교집합·중복 제외",
       [Patch(facecolor=color,label=state) for state,color in STATE_COLOR.items()]+
       [Patch(facecolor="white",edgecolor=COL["WORK"],hatch="////",label="실제 작업 겹침 · 합집합")])
fig.text(.16,.10,"240117: TBM 115분[계획 점검 수행] · CBM 230분[마모량 초과] · 두 작업 동시 41분 → 실제 작업 겹침 304분",
         fontsize=12.5,color=COL["WORK"])
fig.text(.16,.055,"관측 범위에는 시각 누락도 포함된다. 초록은 사후 안정 참고이며 정상 라벨이 아니다. 나머지 13 LOT의 실제 CAL01 작업 겹침은 0분이다.",
         fontsize=11.5,color="#526d80")
finish(fig,"01_품질상태/02_상태_관측길이.png")
table(observation_times)
table(work_overlaps)
table(state_summary)
grade_counts=lots.groupby("GRADE",sort=False).agg(LOT_COUNT=("LOT_NO","size"),MIN_LENGTH=("MINUTES","min"),MAX_LENGTH=("MINUTES","max")).reset_index()
table(grade_counts)
note("**그림 결과:** 240117은 444.32분 범위가 관측되며, TBM 07:40–09:35 115분과 CBM 08:54–12:44 230분이 관측에 겹친다. 두 작업이 동시에 진행된 08:54–09:35 41분을 빼면 실제 겹침은 304분이다. 작업별 시간 345분을 실제 관측 겹침으로 읽지 않는다. 다른 13 LOT은 작업 구간 겹침 0분이며 감지·후속 작업 존재 여부와는 별개다.")
note("**그래프에서 확인:** 240061·240117의 초기 승온과 240058·240114의 종료 전환을 분리했다. 240117의 초기 승온 0–89분 중 작업과 함께 있는 범위는 32–89분, 57분이다. 작업 겹침은 시간 관계이며 온도 변화·정비 효과의 인과 증거가 아니다. 관측 범위는 END−START이며 누락을 제외한 유효 표본 시간과 구분한다.")


## 3. 동일 조건 차이와 전체 공정 그림

**분석 목적:** 동일 강종·안정 참고·동일 경과 분·센서 유효율 ≥90%에서 대상 LOT을 제외한 대조 중앙값과의 차이를 계산한다. 목표값·두께·속도·운전 모드가 없어 운전 조건을 완전히 통제한 비교는 아니다. 대조 1개는 쌍 비교, 2개 이상은 탐색 후보 생성에 사용한다.

**그림 읽기:** 전체 관측 → 색칠된 구간·큰 점 → 같은 번호의 검토 이유 → 함께 볼 자료·확인 계획 → 실제 시각 표 순서로 읽는다. 모든 온도 지속 후보는 연한 붉은 창으로 남기고, 페이지별 최대 차이가 큰 대표 구간 등 최대 3개 이유를 번호로 설명한다. CP는 원본 기준 차이가 가장 큰 펄스를 우선 표시하며 후보 전수는 7절에 남긴다.

| 색 구분 | 왜 이 시간·지점을 보는가? | 함께 볼 자료·다음 검증 |
|---|---|---|
| 붉은색 · 지속 온도 차이 | 대조 2 LOT 이상, 절대 차이 ≥3℃가 3개 1분창 이상 지속 | 원본·대응 OP·인접 TC·품질·작업 시각 및 반대 사례 |
| 청록색 · 쌍 대조 | 강종 내 1개 대조 등 상대 차이를 확인하되 지속 후보 규칙 미충족 | 실제 대조 수와 원본 차이·제어출력·운전 조건 확인 |
| 분홍색 · CP 변화 | 자체 안정 참고 중앙값 대비 원본 차이 ≥0.05 후보의 극값·펄스 | CP-Z4M·TC-Z4·OP-Z4 원본 동시 변화; CP 정의·단위 확인 |
| 노란색 · 초기 승온 | 가열 TC-Z3~6의 초기 대비 후반 상승 형태 | 가열 TC·OP·초기 NaN·수리 후 기록의 선후 관계 |
| 청회색 · 종료 전환 | 끝부분 가열 온도·OP·CP 하강 형태 | 수리 전·관측 종료·타 공정 동시 변화 대조 |
| 검정색 · 반복 파형 | TC-Z9·냉각의 순간 차이가 반복 위상 차이인지 확인 | TC-Z8·TC-Z9·OUT1·OUT2 주기·위상·진폭; OP-Z9는 제공되지 않음 |
| 파란색 · 자체 변동 | 같은 강종 대조가 없으므로 자체 최대 분내 변동 지점을 확인 | 원본·분당 최소–최대·품질·연결 공정; 정상·이상 미판정 |
| 갈색 하단 빗금 · 실제 작업 | CAL01 작업 시각과 관측의 실제 겹침 | 작업 REMARK·시작·종료와 승온·CP·TC/OP 전·중·후 관계 |

**해석 범위:** 색은 검토 이유이며 고장 라벨이 아니다. 3℃·3개 1분창과 CP 기록값 0.05는 탐색 문턱이다. LOT별 경과 분 정렬은 같은 실제 시각의 동시 관측을 의미하지 않으며, 확인 표에는 각 LOT의 실제 시각을 함께 적는다.

In [ ]:
# 같은 강종·안정 참고 상태·같은 경과 분에서 대상 LOT을 뺀 대조만 쓴다.
# 비교할 수 없는 후반 구간을 잘라내거나 대조 중앙값으로 채우지 않는다.
comparison, sensor_rows, episode_rows = {}, [], []
NONCYCLIC = [s for s in TC if s not in ["TC-Z9","TC-OUT1","TC-OUT2"]]
for grade, sub in lots.groupby("GRADE",sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    grid = pd.RangeIndex(max(len(profile[k]) for k in ids))
    for s in SIGNALS:
        values = pd.DataFrame({k:profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고")).reindex(grid) for k in ids})
        for k in ids:
            peers = values.drop(columns=k)
            n = peers.count(axis=1)
            center = peers.median(axis=1)
            delta = (values[k]-center).where(n.ge(1))
            d = pd.DataFrame({"VALUE":values[k],"CENTER":center,"N_PEERS":n,"DELTA":delta})
            comparison[k,s] = d
            v = delta.dropna()
            sensor_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"COMPARE_WINDOWS":len(v),
                                "MEDIAN_DELTA":v.median(),"Q05_DELTA":v.quantile(.05),"Q95_DELTA":v.quantile(.95),
                                "MAX_ABS_DELTA":v.abs().max(),"MIN_PEERS":int(n.loc[v.index].min()) if len(v) else 0})
            if s in NONCYCLIC:
                for a,b in runs(delta.abs().ge(3)&n.ge(2),3):
                    block = d.loc[a:b-1]
                    episode_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"START_MIN":a,"END_MIN":b,
                                         "DURATION_MIN":b-a,"MAX_ABS_DELTA_C":block.DELTA.abs().max(),
                                         "MIN_PEERS":int(block.N_PEERS.min())})
sensor_summary = pd.DataFrame(sensor_rows)
episodes = pd.DataFrame(episode_rows,columns=["LOT","GRADE","SENSOR","START_MIN","END_MIN","DURATION_MIN","MAX_ABS_DELTA_C","MIN_PEERS"])
# CP는 단위가 확인되지 않았으므로 자체 안정 참고 중앙값과 기록값 차이로만 찾는다.
cp_rows, cp_flags = [], {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = raw[k]
    ref = profile[k][("CP-Z4","median")].where(states[k].eq("안정 참고")&quality[k]["CP-Z4"].ge(.9)).median()
    hit = (g["CP-Z4"]-ref).abs().ge(.05)
    cp_flags[k] = pd.Series(profile[k].index.isin(g.loc[hit,"MINUTE"]),index=profile[k].index)
    if hit.any():
        j = (g["CP-Z4"]-ref).abs().idxmax()
        peak = g.loc[j]
        # 5초 초과 시각 공백에서 분리한다. 짧은 공백도 보간하지 않고 실제 기록 행수·최대 간격을 따로 남긴다.
        cluster = ((hit != hit.shift())|g.MEAS_DT.diff().dt.total_seconds().gt(5)).cumsum()
        pulse = g.loc[cluster.eq(cluster.loc[j])&hit]
        cp_rows.append({"LOT":k,"GRADE":r.GRADE,"BASE":ref,"PEAK_MIN":peak.ELAPSED,"PEAK_TIME":peak.MEAS_DT,
                        "PEAK_VALUE":peak["CP-Z4"],"DELTA":peak["CP-Z4"]-ref,
                        "START_MIN":pulse.ELAPSED.min(),"END_MIN":pulse.ELAPSED.max()+1/60,
                        "PULSE_WIDTH_MIN":(pulse.MEAS_DT.max()-pulse.MEAS_DT.min()).total_seconds()/60+1/60,
                        "MAX_RECORD_GAP_S":pulse.MEAS_DT.diff().dt.total_seconds().max(),
                        "RECORDED_SECONDS":len(pulse),"TOTAL_HIT_SECONDS":int(hit.sum()),
                        "TOTAL_HIT_MIN_WINDOWS":int(cp_flags[k].sum())})
cp_candidates = pd.DataFrame(cp_rows,columns=["LOT","GRADE","BASE","PEAK_MIN","PEAK_TIME","PEAK_VALUE","DELTA","START_MIN","END_MIN","PULSE_WIDTH_MIN","MAX_RECORD_GAP_S","RECORDED_SECONDS","TOTAL_HIT_SECONDS","TOTAL_HIT_MIN_WINDOWS"])


In [ ]:
def line(ax, lot, signal, start=None, end=None, clock=False, raw_values=False, envelope=False):
    k = int(lot)
    if raw_values:
        r = lot_meta.loc[k]
        a = max(0, start or 0)
        b = min(r.MINUTES, r.MINUTES if end is None else end)
        # 표시용 1초 격자에만 재배치하여 결측 시각에서 선을 끊는다.
        times = pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)), r.START+pd.Timedelta(seconds=int(b*60)), freq="s")
        p = raw[k].set_index("MEAS_DT").reindex(times)
        x = times if clock else (times-r.START).total_seconds()/60
        y = p[signal].to_numpy()
        low = quality[k][signal].loc[quality[k].index.to_series().between(np.ceil(a),np.floor(b))].lt(.9)
        if low.any():
            marks = low.index[low].to_numpy()
            qx = r.START + pd.to_timedelta(marks,unit="min") if clock else marks
            ax.scatter(qx,np.full(len(marks),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    else:
        p = profile[k]
        p = p.loc[(p.index >= (0 if start is None else start)) & (p.index <= (np.inf if end is None else end))]
        x = lot_meta.loc[k,"START"] + pd.to_timedelta(p.index,unit="min") if clock else p.index.to_numpy()
        y = p[(signal,"median")].to_numpy()
        if envelope:
            ax.fill_between(x,p[(signal,"min")].to_numpy(),p[(signal,"max")].to_numpy(),color=LOT_COLOR[k],alpha=.14,lw=0)
        low = quality[k].loc[p.index,signal].lt(.9)
        if low.any():
            ax.scatter(np.asarray(x)[low],np.full(low.sum(),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    ax.plot(x,y,color=LOT_COLOR[k],lw=1.8)
    ax.set_title(f"{PROCESS[signal]} [{signal}]",loc="left",pad=12)
    ax.set_ylabel(unit(signal),labelpad=10)
    style(ax,"관측 시각[월/일 시:분]" if clock else "관측 시작 후 경과시간[분]")
    ax.xaxis.set_major_locator(MaxNLocator(5)) if not clock else None
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if signal == "CP-Z4M":
        ax.ticklabel_format(axis="y",style="sci",scilimits=(0,0))
    if clock:
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))

QUALITY_HANDLE = Line2D([0],[0],color="#7c8995",marker="|",linestyle="None",markersize=10,label="하단 눈금: 분당 유효 기록 90% 미만")
FOCUS_HANDLE = Patch(facecolor=COL["FOCUS"],alpha=.12,label="검토·확대 시간창")
ENVELOPE_HANDLE = Patch(facecolor="#97b5cc",alpha=.35,label="각 LOT 색 음영: 분당 최소–최대")


In [ ]:
# 색은 검토 이유를 구분한다. 정상·고장 라벨이 아니며, 큰 점은 아래 알파벳 설명에 연결되는 확인 지점이다.
REVIEW_COLOR={"지속 온도 차이":"#c83745","쌍 대조":"#008c95","CP 변화":"#b04482",
              "초기 승온":"#ac7500","종료 전환":"#52738a","반복 파형":"#222222",
              "자체 변동":"#006eaf","실제 작업":"#9a6324"}
PAGE_SIGNALS={"세정·건조":[["CLN-IN"],["TC-Z1","OP-Z1"],["TC-Z2","OP-Z2"]],
              "가열·균열·CP":[["TC-Z3","OP-Z3"],["TC-Z4","OP-Z4"],["CP-Z4","CP-Z4M"]],
              "가열·과시효":[["TC-Z5","OP-Z5"],["TC-Z6","OP-Z6"],["TC-Z7","OP-Z7"],["TC-Z8","OP-Z8"]],
              "과시효·냉각":[["TC-Z9"],["TC-OUT1","TC-OUT2"]]}
page_reviews=[]

def linked_signals(sensor):
    if sensor.startswith("CP"):
        return ["CP-Z4","CP-Z4M","TC-Z4","OP-Z4"]
    if sensor=="CLN-IN":
        return ["CLN-IN"]
    if sensor in ["TC-Z9","TC-OUT1","TC-OUT2"]:
        return ["TC-Z9","TC-OUT1","TC-OUT2"]
    if sensor.startswith("TC-Z"):
        return [sensor,sensor.replace("TC-","OP-")]
    return [sensor]

def next_check(sensor):
    if sensor.startswith("CP"):
        return "CP-Z4M·TC-Z4·OP-Z4 원본 동시 변화와 작업 시각 확인"
    if sensor=="CLN-IN":
        return "CLN-IN 원본·품질을 먼저 보고 건조 TC-Z1/OP-Z1과 대조"
    if sensor in ["TC-Z9","TC-OUT1","TC-OUT2"]:
        return "TC-Z8·TC-Z9·OUT1/OUT2 주기·위상·진폭 대조; OP-Z9 없음"
    return f"{sensor} 원본·{sensor.replace('TC-','OP-')} 동시 변화·인접 TC·작업 시각 대조"

def review_item(kind,k,s,a,b,point,value,why,next_text,related=None):
    return {"KIND":kind,"COLOR":REVIEW_COLOR[kind],"LOT":int(k),"SENSOR":s,
            "START_MIN":max(0,float(a)),"END_MIN":min(float(b),lot_meta.loc[k,"MINUTES"]),
            "POINT_MIN":point,"VALUE":value,"WHY":why,"NEXT":next_text,
            "RELATED":linked_signals(s) if related is None else related}

def grade_reviews(grade,page):
    ids=lots.loc[lots.GRADE.eq(grade),"LOT_NO"].astype(int).tolist()
    signals=[s for row in PAGE_SIGNALS[page] for s in row]
    items=[]
    eps=episodes.loc[episodes.LOT.isin(ids)&episodes.SENSOR.isin(signals)]
    if len(eps):
        e=eps.loc[eps.MAX_ABS_DELTA_C.idxmax()]
        k=int(e.LOT);d=comparison[k,e.SENSOR].loc[int(e.START_MIN):int(e.END_MIN)-1]
        at=d.DELTA.abs().idxmax();delta=d.loc[at,"DELTA"];n=int(d.loc[at,"N_PEERS"])
        items.append(review_item("지속 온도 차이",k,e.SENSOR,e.START_MIN,e.END_MIN,float(at),d.loc[at,"VALUE"],
                     f"대조 {n} LOT 대비 {delta:+.2f}℃; |차이|≥3℃가 {int(e.DURATION_MIN)}개 1분창 지속",
                     next_check(e.SENSOR)))
    else:
        tc_stats=sensor_summary.loc[sensor_summary.LOT.isin(ids)&sensor_summary.SENSOR.isin([s for s in signals if s in TC])&sensor_summary.MAX_ABS_DELTA.notna()]
        if len(tc_stats):
            z=tc_stats.loc[tc_stats.MAX_ABS_DELTA.idxmax()]
            k=int(z.LOT);s=z.SENSOR;d=comparison[k,s];at=d.DELTA.abs().idxmax()
            delta=d.loc[at,"DELTA"];n=int(d.loc[at,"N_PEERS"])
            kind="반복 파형" if s in ["TC-Z9","TC-OUT1","TC-OUT2"] else "쌍 대조"
            why=f"대조 {n} LOT 대비 순간 차이 {delta:+.2f}℃"
            why+="; 반복 위상 차이부터 확인" if kind=="반복 파형" else "; 대조 1개라 다중 대조 후보 판정 제외" if n==1 else "; 지속 후보 규칙은 미충족"
            items.append(review_item(kind,k,s,at-1,at+2,float(at),d.loc[at,"VALUE"],why,next_check(s)))

    if page=="가열·균열·CP":
        cps=cp_candidates.loc[cp_candidates.LOT.isin(ids)]
        if len(cps):
            e=cps.loc[cps.DELTA.abs().idxmax()];k=int(e.LOT)
            why=f"CP 원본 {e.PEAK_VALUE:.4g}, 자체 기준 대비 {e.DELTA:+.4g}; 정의·단위 미확인"
            if work_ranges[k] and e.PEAK_MIN>work_ranges[k][-1][1]:
                why+=f"; 작업 종료 후 {e.PEAK_MIN-work_ranges[k][-1][1]:.1f}분"
            items.append(review_item("CP 변화",k,"CP-Z4",e.START_MIN,e.END_MIN,float(e.PEAK_MIN),e.PEAK_VALUE,
                         why,next_check("CP-Z4")))

    if page!="세정·건조":
        for k in ids:
            sr=state_summary.loc[state_summary.LOT.eq(k)].iloc[0]
            if pd.notna(sr.WARM_END_MIN):
                a=0;b=sr.WARM_END_MIN;kind="초기 승온"
                why=f"가열 TC-Z3~6 초기→80~100분 차이 {sr.RISE_MIN_C:.1f}–{sr.RISE_MAX_C:.1f}℃; 사후 형태 분류"
                nxt="TC-Z3~6·대응 OP·CP를 함께 보고 초기 결측·수리 후 시각 구분"
            elif pd.notna(sr.TAIL_START_MIN):
                a=sr.TAIL_START_MIN;b=lot_meta.loc[k,"MINUTES"];kind="종료 전환"
                why="가열 온도·CP·OP의 끝부분 하강; 안정 참고 구간과 분리"
                nxt="가열 TC·대응 OP·CP 동시 하강과 수리 전·관측 종료 맥락 확인"
            else:
                continue
            items.append(review_item(kind,k,"TC-Z3",a,b,None,None,why,nxt,
                                     related=[s for s in signals if s!="CLN-IN"]))
            break

    for k in ids:
        if work_ranges[k]:
            wr=work_overlaps.loc[work_overlaps.LOT.eq(k)]
            types=" · ".join(f"{r['정비유형']} {r['작업별 겹침[분]']:.0f}분[{r['REMARK']}]" for _,r in wr.iterrows())
            why=types+f"; 중복 제외 {time_meta.loc[k,'작업 겹침 합집합[분]']:.0f}분"
            items.append(review_item("실제 작업",k,signals[0],work_ranges[k][0][0],work_ranges[k][-1][1],
                         None,None,why,"작업 전·중·후 파형과 초기 승온·CP 변화의 선후 관계 확인",related=signals))

    # 강종 내 대조와 사건 맥락이 모두 없으면 자체 변동 지점으로 표시한다. 이상이라고 부르지 않는다.
    if not items:
        choices=[]
        for k in ids:
            for s in [s for s in signals if s in TC]:
                width=(profile[k][(s,"max")]-profile[k][(s,"min")]).where(quality[k][s].ge(.9))
                if width.notna().any():
                    at=width.idxmax();choices.append((float(width.loc[at]),k,s,at))
        width,k,s,at=max(choices)
        kind="반복 파형" if s in ["TC-Z9","TC-OUT1","TC-OUT2"] else "자체 변동"
        items.append(review_item(kind,k,s,at-1,at+2,float(at),profile[k].loc[at,(s,"median")],
                     f"강종 내 대조 없음; 1분 내 최소–최대 폭 {width:.2f}℃; 자체 변화 확인용",next_check(s)))
    assert len(items)<=3,(grade,page,len(items))
    for i,item in enumerate(items,1):
        item["LABEL"]=chr(96+i)
    return items

def short_next_check(item):
    if item["KIND"]=="실제 작업":
        return "작업 전·중·후 / 승온·CP 선후"
    if item["KIND"] in ["초기 승온","종료 전환"]:
        return "가열 TC·대응 OP·CP / 수리·작업 시각"
    sensor=item["SENSOR"]
    if sensor.startswith("CP"):
        return "CP-Z4M·TC-Z4·OP-Z4 원본 / 작업 시각"
    if sensor=="CLN-IN":
        return "CLN-IN 원본·TC-Z1·OP-Z1"
    if sensor in ["TC-Z9","TC-OUT1","TC-OUT2"]:
        return "Z8·Z9·OUT1/2 주기·위상·진폭"
    return f"{sensor} 원본·{sensor.replace('TC-','OP-')} / 인접 TC·작업 시각"

def grade_page(grade,page,name):
    ids=lots.loc[lots.GRADE.eq(grade),"LOT_NO"].astype(int).tolist()
    rows=PAGE_SIGNALS[page];signals=[s for row in rows for s in row]
    reviews=grade_reviews(grade,page)
    note(f"**그림 목적 — {grade} {page}:** 전체 관측을 유지하면서 색·(a)~(c) 표시로 먼저 볼 구간을 찾는다. 알파벳은 실제 확인 위치 위에 있으며 아래 같은 표시의 한 줄 설명과 연결한다. 붉은 온도 후보는 같은 강종·안정 참고·동일 경과 분·유효율 ≥90%·대조 2 LOT 이상에서 |차이|≥3℃가 3개 1분창 이상 지속된 경우다. 쌍 대조·자체 변동·반복 위상·승온·종료·작업 맥락을 고장으로 확정하지 않는다.")
    height=24 if len(rows)==4 else 20 if len(rows)==3 else 16
    fig=plt.figure(figsize=(30,height))
    gs=fig.add_gridspec(len(rows),2)
    fig.subplots_adjust(left=.09,right=.97,top=.79,bottom=.27,wspace=.30,hspace=.95)
    eps=episodes.loc[episodes.LOT.isin(ids)&episodes.SENSOR.isin(signals)]
    xmax=lots.loc[lots.LOT_NO.isin(ids),"MINUTES"].max()+2
    callouts=[]
    for i,row in enumerate(rows):
        for j,s in enumerate(row):
            ax=fig.add_subplot(gs[i,:] if len(row)==1 else gs[i,j])
            for k in ids:line(ax,k,s,envelope=True)
            tc=s.replace("OP-","TC-") if s in OP else s
            intervals=[(r.START_MIN,r.END_MIN) for r in eps.loc[eps.SENSOR.eq(tc)].itertuples()]
            for a,b in merge_ranges(intervals):
                ax.axvspan(a,b,color=REVIEW_COLOR["지속 온도 차이"],alpha=.07,lw=0)
            used_positions=[]
            for item in reviews:
                if s not in item["RELATED"]:continue
                a=item["START_MIN"];b=item["END_MIN"];color=item["COLOR"];kind=item["KIND"]
                if kind=="실제 작업":
                    for start,end in work_ranges[item["LOT"]]:
                        ax.axvspan(start,end,ymin=0,ymax=.065,color=color,alpha=.30,lw=0,hatch="////")
                else:
                    ax.axvspan(a,b,color=color,alpha=.10,lw=0)
                if item["POINT_MIN"] is not None:
                    highlight=profile[item["LOT"]][(s,"median")]
                    inside=(highlight.index>=a)&(highlight.index<=b)
                    ax.plot(highlight.index,highlight.where(inside),color=color,lw=2.7,zorder=7)
                    ax.axvline(item["POINT_MIN"],color=color,ls=":",lw=1.2,alpha=.85)
                    if s==item["SENSOR"]:
                        ax.scatter([item["POINT_MIN"]],[item["VALUE"]],s=65,
                                   facecolors="white",edgecolors=color,linewidths=1.8,zorder=8)
                # 알파벳은 파형 위의 별도 여백에 둔다. 가까운 표시는 옆으로 벌려 실제 위치와 선으로 연결한다.
                focus_at=item["POINT_MIN"] if item["POINT_MIN"] is not None else (a+b)/2
                focus_x=focus_at/xmax
                candidates=[np.clip(focus_x+offset,.085 if s=="CP-Z4M" else .04,.96) for offset in [0,-.09,.09,-.18,.18,-.27,.27]]
                label_x=next(x for x in candidates if all(abs(x-other)>=.07 for other in used_positions))
                used_positions.append(label_x)
                ax.plot([focus_x,label_x],[1.005,1.05],transform=ax.transAxes,
                        color=color,lw=.9,clip_on=False)
                label=ax.text(label_x,1.052,f"({item['LABEL']})",transform=ax.transAxes,
                              ha="center",va="bottom",fontsize=16,fontweight="bold",color=color,clip_on=False)
                callouts.append((ax,label))
            ax.set_xlim(0,xmax)
            ax.set_title(f"{PROCESS[s]} [{s}]",loc="left",pad=43,fontsize=18)
            ax.tick_params(labelsize=14)
            ax.set_xlabel("관측 시작 후 경과시간[분]",fontsize=15,labelpad=10)
            ax.yaxis.label.set_fontsize(15)

    kinds=list(dict.fromkeys(r["KIND"] for r in reviews))
    if len(eps) and "지속 온도 차이" not in kinds:kinds.insert(0,"지속 온도 차이")
    extra=[Patch(facecolor=REVIEW_COLOR[k],alpha=.20,label=f"{k} · 검토 위치") for k in kinds]
    header(fig,f"{grade} {page} 비교","(a)~(c): 실제 확인 위치와 아래 한 줄 설명 연결 · 연한 색: 검토 시간창 · 강조선·큰 점: 확인 지점 · 선: 1분 중앙값",
           lot_handles(ids)+extra+[ENVELOPE_HANDLE,QUALITY_HANDLE],ncol=4)
    fig._suptitle.set_fontsize(26)
    fig.texts[-1].set_fontsize(15);fig.texts[-1].set_position((.035,.94))
    fig.legends[0].set_bbox_to_anchor((.031,.90))
    for text in fig.legends[0].get_texts():text.set_fontsize(14)
    for i,item in enumerate(reviews):
        a=item["START_MIN"];b=item["END_MIN"]
        caption=f"({item['LABEL']}) {item['LOT']} [{item['SENSOR']}] {a:.1f}–{b:.1f}분 | {item['WHY']} | 다시 보기: {short_next_check(item)}"
        fig.text(.09,.18-i*.055,caption,fontsize=15,color=item["COLOR"],ha="left",va="top")
    fig.set_dpi(160);fig.canvas.draw()
    renderer=fig.canvas.get_renderer()
    assert all(not text_box(label,renderer).overlaps(ax.get_window_extent(renderer))
               for ax,label in callouts),"알파벳 표시가 실제 파형 영역을 가림"
    finish(fig,name)

    note(f"**그림 결과:** {grade} {len(ids)} LOT의 {page} 전체 관측을 보존하고 {len(reviews)}개 검토 이유를 (a)~(c)로 표시했다. 연한 붉은 창은 이 페이지 센서의 지속 온도 후보 전수, 알파벳·큰 점은 우선 확인 지점이다. 대조 없는 후반은 파형에 남지만 대조 차이는 계산하지 않는다.")
    review_table=[]
    for item in reviews:
        k=item["LOT"];a=item["START_MIN"];b=item["END_MIN"]
        start=lot_meta.loc[k,"START"]+pd.Timedelta(minutes=a)
        end=lot_meta.loc[k,"START"]+pd.Timedelta(minutes=b)
        point=lot_meta.loc[k,"START"]+pd.Timedelta(minutes=item["POINT_MIN"]) if item["POINT_MIN"] is not None else pd.NaT
        row={"강종":grade,"그림 공정":page,"표시":f"({item['LABEL']})","색 구분":item["KIND"],"LOT":k,"센서":item["SENSOR"],
             "시작 경과[분]":a,"종료 경과[분]":b,"표시 범위[분]":b-a,
             "실제 시작":start,"실제 종료":end,"확인 지점":point,
             "검토 이유":item["WHY"],"함께 볼 자료·검증 계획":item["NEXT"],"판정":"검토 대상 · 고장 미확정"}
        review_table.append(row);page_reviews.append(row)
        note(f"**({item['LABEL']}) 확인:** {item['WHY']}. **다시 볼 자료:** {item['NEXT']}. 실제 시각은 {start:%m/%d %H:%M:%S}–{end:%H:%M:%S}이며, 경과 분으로 정렬한 다른 LOT과 실제 같은 시각을 뜻하지 않는다.")
    table(pd.DataFrame(review_table))
    if len(eps):
        note("**온도 후보 전수:** 이 페이지 관련 센서의 모든 지속 구간을 남긴다. 알파벳 지점은 최대 절대 차이가 큰 대표 구간이며 나머지 후보를 삭제하지 않는다.")
        table(eps)
    stats=sensor_summary.loc[sensor_summary.LOT.isin(ids)&sensor_summary.SENSOR.isin(signals)]
    for group,label,suffix in [(TC,"온도","℃"),(OP,"제어출력","%p"),(["CP-Z4","CP-Z4M"],"CP","원본 기록값")]:
        d=stats.loc[stats.SENSOR.isin(group)&stats.MAX_ABS_DELTA.notna()]
        if len(d):
            z=d.loc[d.MAX_ABS_DELTA.idxmax()];data=comparison[int(z.LOT),z.SENSOR]
            at=data.DELTA.abs().idxmax();signed=data.loc[at,"DELTA"];peers=int(data.loc[at,"N_PEERS"])
            note(f"**{label} 대조 결과:** {int(z.LOT)} [{z.SENSOR}] {at:.0f}분 대상−대조 중앙값 {signed:+.6g}{suffix}, 대조 {peers} LOT다. 전체 비교 가능 {int(z.COMPARE_WINDOWS)}개 1분창이며, 대조 수가 다른 최대 차이와 지속 후보를 구분한다.")
    if len(ids)==1:
        note("**대조 한계:** 같은 강종의 다른 LOT이 없어 표시한 자체 변화·공정 흐름·작업 시각으로만 검토한다. 정상·이상 판정에는 운전 모드·목표값·두께·속도 등 추가 조건이 필요하다.")
    note("**이 그림으로 좁히는 범위:** 색·알파벳·표의 LOT·센서·실제 시각을 03의 원본·반대 사례·동시 변화 검증과 연결한다. 색은 검토 이유를 나타내며 정상·고장 확정 색이 아니다. TC-Z9·냉각은 주기·위상부터 대조하고, CP-Z4M에는 정의 미확인 상태를 유지한다.")
    table(stats)


### SGCC 전체 공정

**목적:** SGCC의 모든 관측 LOT을 세정·건조 → 가열·균열·CP → 가열·과시효 → 과시효·냉각 순서로 읽는다. 전체 관측에서 강조된 후보는 03에서 원본·반대 사례로 확대 검증한다.

In [ ]:
grade_page("SGCC","세정·건조","02_강종별전체/01_SGCC/01_SGCC_세정_건조_전체.png")

In [ ]:
grade_page("SGCC","가열·균열·CP","02_강종별전체/01_SGCC/02_SGCC_가열_균열_CP_전체.png")

In [ ]:
grade_page("SGCC","가열·과시효","02_강종별전체/01_SGCC/03_SGCC_가열_과시효_전체.png")

In [ ]:
grade_page("SGCC","과시효·냉각","02_강종별전체/01_SGCC/04_SGCC_과시효_냉각_전체.png")

### SPFH590 전체 공정

**목적:** SPFH590의 모든 관측 LOT을 세정·건조 → 가열·균열·CP → 가열·과시효 → 과시효·냉각 순서로 읽는다. 전체 관측에서 강조된 후보는 03에서 원본·반대 사례로 확대 검증한다.

In [ ]:
grade_page("SPFH590","세정·건조","02_강종별전체/02_SPFH590/01_SPFH590_세정_건조_전체.png")

In [ ]:
grade_page("SPFH590","가열·균열·CP","02_강종별전체/02_SPFH590/02_SPFH590_가열_균열_CP_전체.png")

In [ ]:
grade_page("SPFH590","가열·과시효","02_강종별전체/02_SPFH590/03_SPFH590_가열_과시효_전체.png")

In [ ]:
grade_page("SPFH590","과시효·냉각","02_강종별전체/02_SPFH590/04_SPFH590_과시효_냉각_전체.png")

### SPCE 전체 공정

**목적:** SPCE의 모든 관측 LOT을 세정·건조 → 가열·균열·CP → 가열·과시효 → 과시효·냉각 순서로 읽는다. 전체 관측에서 강조된 후보는 03에서 원본·반대 사례로 확대 검증한다.

In [ ]:
grade_page("SPCE","세정·건조","02_강종별전체/03_SPCE/01_SPCE_세정_건조_전체.png")

In [ ]:
grade_page("SPCE","가열·균열·CP","02_강종별전체/03_SPCE/02_SPCE_가열_균열_CP_전체.png")

In [ ]:
grade_page("SPCE","가열·과시효","02_강종별전체/03_SPCE/03_SPCE_가열_과시효_전체.png")

In [ ]:
grade_page("SPCE","과시효·냉각","02_강종별전체/03_SPCE/04_SPCE_과시효_냉각_전체.png")

### SPCEN 전체 공정

**목적:** SPCEN의 모든 관측 LOT을 세정·건조 → 가열·균열·CP → 가열·과시효 → 과시효·냉각 순서로 읽는다. 전체 관측에서 강조된 후보는 03에서 원본·반대 사례로 확대 검증한다.

In [ ]:
grade_page("SPCEN","세정·건조","02_강종별전체/04_SPCEN/01_SPCEN_세정_건조_전체.png")

In [ ]:
grade_page("SPCEN","가열·균열·CP","02_강종별전체/04_SPCEN/02_SPCEN_가열_균열_CP_전체.png")

In [ ]:
grade_page("SPCEN","가열·과시효","02_강종별전체/04_SPCEN/03_SPCEN_가열_과시효_전체.png")

In [ ]:
grade_page("SPCEN","과시효·냉각","02_강종별전체/04_SPCEN/04_SPCEN_과시효_냉각_전체.png")

### SPHC 전체 공정

**목적:** SPHC의 모든 관측 LOT을 세정·건조 → 가열·균열·CP → 가열·과시효 → 과시효·냉각 순서로 읽는다. 전체 관측에서 강조된 후보는 03에서 원본·반대 사례로 확대 검증한다.

In [ ]:
grade_page("SPHC","세정·건조","02_강종별전체/05_SPHC/01_SPHC_세정_건조_전체.png")

In [ ]:
grade_page("SPHC","가열·균열·CP","02_강종별전체/05_SPHC/02_SPHC_가열_균열_CP_전체.png")

In [ ]:
grade_page("SPHC","가열·과시효","02_강종별전체/05_SPHC/03_SPHC_가열_과시효_전체.png")

In [ ]:
grade_page("SPHC","과시효·냉각","02_강종별전체/05_SPHC/04_SPHC_과시효_냉각_전체.png")

### DP590 전체 공정

**목적:** DP590의 모든 관측 LOT을 세정·건조 → 가열·균열·CP → 가열·과시효 → 과시효·냉각 순서로 읽는다. 전체 관측에서 강조된 후보는 03에서 원본·반대 사례로 확대 검증한다.

In [ ]:
grade_page("DP590","세정·건조","02_강종별전체/06_DP590/01_DP590_세정_건조_전체.png")

In [ ]:
grade_page("DP590","가열·균열·CP","02_강종별전체/06_DP590/02_DP590_가열_균열_CP_전체.png")

In [ ]:
grade_page("DP590","가열·과시효","02_강종별전체/06_DP590/03_DP590_가열_과시효_전체.png")

In [ ]:
grade_page("DP590","과시효·냉각","02_강종별전체/06_DP590/04_DP590_과시효_냉각_전체.png")

### SPCD 전체 공정

**목적:** SPCD의 모든 관측 LOT을 세정·건조 → 가열·균열·CP → 가열·과시효 → 과시효·냉각 순서로 읽는다. 전체 관측에서 강조된 후보는 03에서 원본·반대 사례로 확대 검증한다.

In [ ]:
grade_page("SPCD","세정·건조","02_강종별전체/07_SPCD/01_SPCD_세정_건조_전체.png")

In [ ]:
grade_page("SPCD","가열·균열·CP","02_강종별전체/07_SPCD/02_SPCD_가열_균열_CP_전체.png")

In [ ]:
grade_page("SPCD","가열·과시효","02_강종별전체/07_SPCD/03_SPCD_가열_과시효_전체.png")

In [ ]:
grade_page("SPCD","과시효·냉각","02_강종별전체/07_SPCD/04_SPCD_과시효_냉각_전체.png")

## 4. 수준·변동·추세

**목적:** 전체 LOT의 평균 하나로 승온·안정·종료를 섞지 않는다. 상태별 유효 분의 중앙값·IQR·시간 기울기·끝 10개 분 대비 첫 10개 분 차이를 실제 단위로 기록한다. 기울기와 차이는 기술 통계이며 독립 표본에 대한 유의성 검정이 아니다.

In [ ]:
GRADE_COLOR={"SGCC":"#455a64","SPFH590":"#d55e00","SPCE":"#0072b2","SPCEN":"#e69f00","SPHC":"#009e73","DP590":"#a23b57","SPCD":"#7b2cbf"}
state_stats=[]
for r in lots.itertuples():
    k=int(r.LOT_NO)
    for state in states[k].unique():
        for s in SIGNALS:
            y=profile[k][(s,"median")].where(states[k].eq(state)&quality[k][s].ge(.9)).dropna()
            state_stats.append({"LOT":k,"GRADE":r.GRADE,"STATE":state,"SENSOR":s,"N_WINDOWS":len(y),"MEDIAN":y.median(),
                                "IQR":y.quantile(.75)-y.quantile(.25),"FIRST_LAST10_DELTA":y.tail(10).median()-y.head(10).median() if len(y)>=20 else np.nan,
                                "SLOPE_PER_MIN":np.polyfit(y.index.to_numpy(),y.to_numpy(),1)[0] if len(y)>=20 else np.nan})
state_stats=pd.DataFrame(state_stats)
chosen=[("TC-Z3","MEDIAN","가열·균열 [TC-Z3] · 수준","온도 중앙값[℃]"),
        ("TC-Z3","IQR","가열·균열 [TC-Z3] · 변동","온도 IQR[℃]"),
        ("TC-Z3","FIRST_LAST10_DELTA","가열·균열 [TC-Z3] · 시간 변화","후반–초반 온도[℃]"),
        ("OP-Z3","FIRST_LAST10_DELTA","가열·균열 [OP-Z3] · 시간 변화","후반–초반 제어출력[%p]")]
fig,axs=plt.subplots(2,2,figsize=(24,13))
fig.subplots_adjust(left=.14,right=.97,top=.77,bottom=.10,wspace=.36,hspace=.62)
for ax,(s,column,title,ylabel) in zip(axs.flat,chosen):
    d=state_stats.loc[state_stats.SENSOR.eq(s)&state_stats.STATE.eq("안정 참고")].set_index("LOT").reindex(lots.LOT_NO)
    ax.barh(range(len(lots)),d[column],color=[GRADE_COLOR[g] for g in lots.GRADE],height=.6)
    ax.set_yticks(range(len(lots)),[str(k) for k in lots.LOT_NO]);ax.invert_yaxis()
    ax.set_title(title,loc="left",pad=12);ax.set_ylabel("LOT[관측 시작순]",labelpad=10)
    style(ax,ylabel)
header(fig,"안정 참고 통계 비교","상태별·센서별 유효율 ≥90% 분만 사용; 안정 참고는 실제 정상 라벨이 아님",
       [Patch(facecolor=col,label=grade) for grade,col in GRADE_COLOR.items()],ncol=4)
finish(fig,"03_통계대조/01_안정참고_수준변동추세.png")
tc3=state_stats.loc[state_stats.SENSOR.eq("TC-Z3")&state_stats.STATE.eq("안정 참고")].set_index("LOT")
low,high=tc3.MEDIAN.idxmin(),tc3.MEDIAN.idxmax()
wide=tc3.IQR.idxmax();trend=tc3.FIRST_LAST10_DELTA.abs().idxmax()
note(f"**그림 결과:** 안정 참고 TC-Z3 중앙값은 {int(low)}·{tc3.loc[low,'GRADE']} {tc3.loc[low,'MEDIAN']:.2f}℃부터 {int(high)}·{tc3.loc[high,'GRADE']} {tc3.loc[high,'MEDIAN']:.2f}℃까지다. 최대 IQR은 {int(wide)}의 {tc3.loc[wide,'IQR']:.2f}℃, 후반−초반 차이의 최대 절대값은 {int(trend)}의 {tc3.loc[trend,'FIRST_LAST10_DELTA']:+.2f}℃다. 전체 수준 차이·변동·시간 추세를 분리해 볼 수 있다. 다음 동일 강종 대조에서 비교 수와 차이를 다시 확인하며 이 순위를 고장 순위로 해석하지 않는다.")
for group in ["세정","건조","가열·균열","과시효","냉각"]:
    sensors=[s for s in SIGNALS if PROCESS[s]==group]
    note(f"**{group} 전수 통계:** MEDIAN·IQR은 축 표시 단위, SLOPE_PER_MIN은 해당 단위/분, OP 차이는 %p, CP는 원본 기록값 차이다.")
    table(state_stats.loc[state_stats.SENSOR.isin(sensors)])
note("**그래프에서 확인:** 강종별 수준이 다르고 같은 강종 안에서도 관측 길이·추세가 다르다. 이 차트는 상태별 기술 통계다. 수준이 높은 강종을 고장 후보로 우선하지 않고 아래 동일 강종·동일 시간의 차이와 대조 수로 판단한다.")

## 5. 동일 조건 대조 범위

**목적:** 전체 관측에서 실제 온도 차이와 동시에 사용 가능한 대조 LOT 수를 함께 본다. 대조가 0개인 후반은 차이를 계산하지 않으며, 1개 대조와 2개 이상 대조를 구분한다.

In [ ]:
peer_rows=[]
for r in lots.itertuples():
    k=int(r.LOT_NO)
    for s in TC:
        d=comparison[k,s].reindex(profile[k].index)
        valid=d.VALUE.notna()
        peer_rows.append({"LOT":k,"GRADE":r.GRADE,"SENSOR":s,"OWN_VALID_WINDOWS":int(valid.sum()),
                          "NO_PEER_WINDOWS":int((valid&d.N_PEERS.eq(0)).sum()),
                          "ONE_PEER_WINDOWS":int((valid&d.N_PEERS.eq(1)).sum()),
                          "MULTI_PEER_WINDOWS":int((valid&d.N_PEERS.ge(2)).sum())})
peer_summary=pd.DataFrame(peer_rows)
for grade in ["SPHC","SPCE"]:
    ids=lots.loc[lots.GRADE.eq(grade),"LOT_NO"].astype(int).tolist()
    fig,axs=plt.subplots(2,2,figsize=(24,11))
    fig.subplots_adjust(left=.10,right=.97,top=.77,bottom=.12,wspace=.28,hspace=.67)
    for j,s in enumerate(["TC-Z3","TC-Z4"]):
        for k in ids:
            d=comparison[k,s]
            axs[0,j].plot(d.index,d.DELTA,color=LOT_COLOR[k],lw=1.7)
            axs[1,j].step(d.index,d.N_PEERS.where(d.VALUE.notna()),color=LOT_COLOR[k],where="post",lw=1.7)
        axs[0,j].axhline(3,color=COL["FOCUS"],ls=":",lw=1)
        axs[0,j].axhline(-3,color=COL["FOCUS"],ls=":",lw=1)
        axs[0,j].set_title(f"가열·균열 [{s}] · 같은 강종 대조 차이",loc="left",pad=12)
        axs[0,j].set_ylabel("대상–대조 중앙값[℃]",labelpad=10)
        axs[1,j].set_title(f"가열·균열 [{s}] · 대조 LOT 수",loc="left",pad=12)
        axs[1,j].set_ylabel("유효 대조 LOT[개]",labelpad=10)
        axs[1,j].set_yticks(range(len(ids)));axs[1,j].set_ylim(-.15,len(ids)-.7)
    for ax in axs.flat:
        style(ax);ax.set_xlim(0,lots.loc[lots.LOT_NO.isin(ids),"MINUTES"].max()+2)
    header(fig,f"{grade} 온도 차이·대조 수","같은 강종·안정 참고·같은 경과 분; 대상 LOT을 제외한 기준",
           lot_handles(ids)+[Line2D([0],[0],color=COL["FOCUS"],ls=":",label="탐색 문턱 ±3℃")],ncol=4)
    finish(fig,f"03_통계대조/02_{grade}_실제차이_대조수.png")
    for k in ids:
        for signal in ["TC-Z3","TC-Z4"]:
            d=comparison[k,signal];v=d.DELTA.dropna()
            if len(v):
                note(f"**그림 결과:** {k} [{signal}] 대조 차이 {v.min():+.2f}–{v.max():+.2f}℃, 대조 2 LOT 이상인 유효 분 {int((d.VALUE.notna()&d.N_PEERS.ge(2)).sum())}개다. 차이 계산의 마지막 시각은 경과 {v.index[-1]:.0f}분이며 이후 빈 선은 대상 관측 종료 또는 대조·상태·품질 조건 부족이다.")
    table(peer_summary.loc[peer_summary.GRADE.eq(grade)&peer_summary.SENSOR.isin(["TC-Z3","TC-Z4"])])
    table(sensor_summary.loc[sensor_summary.GRADE.eq(grade)&sensor_summary.SENSOR.isin(["TC-Z3","TC-Z4"])])
    note(f"**결과:** {grade}의 온도 차이는 대조가 있는 시간에만 계산된다. 전체 길이를 보존하여 후반 대조 부족과 초기 상태 제외를 확인할 수 있다. 두 개 이상의 대조가 있는 분만 아래 지속 후보 생성에 사용한다.")
table(peer_summary.loc[~peer_summary.GRADE.isin(["SPHC","SPCE"])])
note("**다른 강종의 결과:** SPFH590·SPCEN은 최대 1개 대조, SGCC·DP590·SPCD는 0개 대조다. 비교 불가는 정상 판정이 아니라 근거 부족이며 자체 시간 변화·사건 맥락 검증으로 전달한다.")

## 6. 연관성 단서: TC·OP와 시차

**목적:** 같은 상태 내 온도와 대응 제어출력의 수준 상관과 1분 차분 상관을 비교한다. 연속 유효 분에서만 차분을 계산하고 TC-Z9에는 OP를 붙이지 않는다. ΔOP(t−lag)와 ΔTC(t)의 Spearman 상관을 0–10분에서 계산하되 공정 지연·원인·고장 예측력으로 확정하지 않는다. 짧은 표본과 자기상관 때문에 유의확률을 LOT 수처럼 해석하지 않는다.

In [ ]:
corr_rows,lag_rows=[],[]
for r in lots.itertuples():
    k=int(r.LOT_NO);p=profile[k]
    for z in range(1,9):
        tc,op=f"TC-Z{z}",f"OP-Z{z}"
        x,y=p[(tc,"median")],p[(op,"median")]
        valid=states[k].eq("안정 참고")&quality[k][tc].ge(.9)&quality[k][op].ge(.9)
        pair=pd.DataFrame({"TC":x.where(valid),"OP":y.where(valid)}).dropna()
        delta=pd.DataFrame({"TC":x.diff().where(valid&valid.shift(1,fill_value=False)),
                            "OP":y.diff().where(valid&valid.shift(1,fill_value=False))})
        d=delta.dropna()
        level=pair.corr(method="spearman").iloc[0,1] if len(pair)>=30 and pair.nunique().min()>1 else np.nan
        diff=d.corr(method="spearman").iloc[0,1] if len(d)>=30 and d.nunique().min()>1 else np.nan
        corr_rows.append({"LOT":k,"GRADE":r.GRADE,"ZONE":z,"N_LEVEL":len(pair),"LEVEL_RHO":level,"N_DIFF":len(d),"DIFF_RHO":diff})
        for lag in range(11):
            shifted=pd.concat([delta.TC,delta.OP.shift(lag)],axis=1).dropna()
            rho=shifted.corr(method="spearman").iloc[0,1] if len(shifted)>=30 and shifted.nunique().min()>1 else np.nan
            lag_rows.append({"LOT":k,"ZONE":z,"LAG_MIN":lag,"N_WINDOWS":len(shifted),"RHO":rho})
corr_table=pd.DataFrame(corr_rows);lag_table=pd.DataFrame(lag_rows)
fig,axs=plt.subplots(1,2,figsize=(24,11))
fig.subplots_adjust(left=.14,right=.89,top=.79,bottom=.14,wspace=.52)
for ax,column,title in zip(axs,["LEVEL_RHO","DIFF_RHO"],["TC·OP 수준 상관","TC·OP 1분 차분 상관"]):
    m=corr_table.pivot(index="LOT",columns="ZONE",values=column).reindex(lots.LOT_NO)
    im=ax.imshow(m.to_numpy(),aspect="auto",cmap="RdBu_r",vmin=-1,vmax=1)
    ax.set_xticks(range(8),[f"Z{z}" for z in range(1,9)])
    ax.set_yticks(range(len(lots)),[f"{r.LOT_NO} · {r.GRADE}" for r in lots.itertuples()])
    ax.set_title(title,loc="left",pad=12)
    ax.set_xlabel("동일 존 [TC-Zn ↔ OP-Zn]",labelpad=10);ax.set_ylabel("LOT · 강종",labelpad=10)
cax=fig.add_axes([.93,.25,.015,.48])
cb=fig.colorbar(im,cax=cax);cb.set_label("Spearman 상관계수[ρ]",labelpad=12)
header(fig,"온도·출력 상관 대조","같은 상태 내 유효 관측; 차분은 연속 유효 분에만 계산; 빈칸은 표본·변동 부족")
finish(fig,"04_연관성/01_TC_OP_수준_차분상관.png")
for column,label,ncolumn in [("LEVEL_RHO","수준","N_LEVEL"),("DIFF_RHO","1분 차분","N_DIFF")]:
    d=corr_table.loc[corr_table[column].notna()]
    z=d.loc[d[column].abs().idxmax()]
    note(f"**그림 결과 — {label}:** 계산 가능 {len(d)}개 LOT·존 조합 중 최대 절대 상관은 {int(z.LOT)} Z{int(z.ZONE)}의 ρ={z[column]:+.3f}, 유효 {int(z[ncolumn])}분이다. 파랑은 음의 상관, 빨강은 양의 상관이며 크기와 방향을 함께 읽는다.")
contrast=(corr_table.LEVEL_RHO-corr_table.DIFF_RHO).abs()
z=corr_table.loc[contrast.idxmax()]
note(f"**대조 해석:** 수준↔차분 상관 차이가 가장 큰 {int(z.LOT)} Z{int(z.ZONE)}는 {z.LEVEL_RHO:+.3f} → {z.DIFF_RHO:+.3f}다. 수준 상관만으로 연관성을 정하지 않고 연속 분 변화와 아래 ΔOP(t−lag)↔ΔTC(t) 시차를 추가 확인한다. 제어 피드백·공통 운전 조건이 남아 인과·고장 예측력은 확정하지 않는다.")
table(corr_table)
best=lag_table.loc[lag_table.RHO.notna()].copy()
best=best.loc[best.groupby(["LOT","ZONE"]).RHO.apply(lambda s:s.abs().idxmax()).to_numpy()]
table(best)
change=(corr_table.LEVEL_RHO-corr_table.DIFF_RHO).abs()
note(f"**결과:** 수준·차분 상관의 절대 차이 중앙값은 {change.median():.3f}이다. 추세와 제어 반응의 영향을 함께 확인할 단서다. 최대 상관 시차는 탐색 결과이며 11개 시차에서 골랐다는 선택 효과가 있다. 미래 특징이나 고장 라벨에 이 값을 바로 쓰지 않는다.")

## 7. 후보 전수와 문턱 민감도

**목적:** 후보가 특정 문턱이나 작은 표준편차 점수에 의존하는지 확인한다. 작은 MAD를 나눠 큰 Z 점수를 만들지 않고 온도 차이·지속·대조 수를 직접 쓴다. CP는 단위·정의 부족을 표시하며 원본 극값을 모두 검증 대상으로 전달한다.

In [ ]:
sensitivity=[]
for threshold in [1,3,5,10]:
    for duration in [1,3,5]:
        spans=[];ids=set()
        for (k,s),d in comparison.items():
            if s not in NONCYCLIC:continue
            rr=runs(d.DELTA.abs().ge(threshold)&d.N_PEERS.ge(2),duration)
            if rr:ids.add(k)
            spans.extend(rr)
        sensitivity.append({"DELTA_C":threshold,"MIN_DURATION":duration,"N_LOTS":len(ids),"N_SENSOR_EPISODES":len(spans)})
sensitivity=pd.DataFrame(sensitivity)
fig,axs=plt.subplots(1,2,figsize=(23,8))
fig.subplots_adjust(left=.09,right=.95,top=.77,bottom=.20,wspace=.30)
for ax,value,title in zip(axs,["N_LOTS","N_SENSOR_EPISODES"],["조건별 후보 LOT 수","조건별 센서 구간 수"]):
    m=sensitivity.pivot(index="MIN_DURATION",columns="DELTA_C",values=value)
    im=ax.imshow(m.to_numpy(),cmap="Blues",aspect="auto",vmin=0)
    ax.set_xticks(range(4),[1,3,5,10]);ax.set_yticks(range(3),[1,3,5])
    ax.set_xlabel("최소 절대 온도 차이[℃]",labelpad=10);ax.set_ylabel("최소 연속 시간[분]",labelpad=10)
    ax.set_title(title,loc="left",pad=12)
    for i in range(3):
        for j in range(4):
            ax.text(j,i,str(int(m.iloc[i,j])),ha="center",va="center",color="white" if m.iloc[i,j]>m.to_numpy().max()*.55 else "#17344d",fontsize=13)
header(fig,"후보 문턱 민감도","같은 강종·상태·유효율 ≥90%·대조 2 LOT 이상; 각 칸 숫자는 개수; 각 패널 안에서 진하기를 비교",
       [Patch(facecolor="#c6dbef",label="밝은 파랑: 후보 적음"),Patch(facecolor="#08519c",label="어두운 파랑: 후보 많음")])
finish(fig,"05_후보선정/01_문턱_민감도.png")
for duration in [1,3,5]:
    d=sensitivity.loc[sensitivity.MIN_DURATION.eq(duration)].set_index("DELTA_C")
    note(f"**그림 결과 — {duration}분 지속:** 차이 문턱 1→10℃에서 후보 LOT {int(d.loc[1,'N_LOTS'])}→{int(d.loc[10,'N_LOTS'])}개, 센서 구간 {int(d.loc[1,'N_SENSOR_EPISODES'])}→{int(d.loc[10,'N_SENSOR_EPISODES'])}개다. 구간 수는 큰 문턱에서 하나의 구간이 갈라질 수 있어 단순 단조 감소를 가정하지 않는다.")
table(sensitivity);table(episodes);table(cp_candidates)
note(f"**결과:** 주 규칙 3℃·3분의 온도 센서 구간은 {len(episodes)}개, 관련 LOT은 {episodes.LOT.nunique()}개다. CP 원본 기준 차이 ≥0.05 LOT은 {len(cp_candidates)}개다. 후보 수 변화는 위 표에 있다. 문턱 하나의 결과를 확정 고장 목록으로 사용하지 않는다.")

## 8. 03에 전달할 LOT·공정·시간창

**목적:** 통계 후보와 수리·감지·작업 맥락을 함께 남기고, 후보가 없는 LOT도 대조 역할을 확인한다. 우선순위 점수 하나로 다른 근거를 합쳐 지우지 않는다. 전체 전수 표와 강종별 반대 사례가 가설 검증의 입력이다.

In [ ]:
selection=context.copy()
selection=selection.merge(observation_times[["LOT","작업 겹침 합집합[분]","작업별 겹침 단순합[분]","중복 집계분[분]"]],on="LOT",validate="one_to_one")
selection["TEMP_EPISODES"]=selection.LOT.map(episodes.groupby("LOT").size()).fillna(0).astype(int)
selection["CP_REVIEW"]=selection.LOT.isin(cp_candidates.LOT)
selection=selection.merge(state_summary[["LOT","WARM_END_MIN","TAIL_START_MIN"]],on="LOT",validate="one_to_one")
selection["REVIEW_REASON"]=selection.apply(lambda r:" · ".join(
    (["초기 승온"] if pd.notna(r.WARM_END_MIN) else [])+
    (["종료 전환"] if pd.notna(r.TAIL_START_MIN) else [])+
    ([f"온도 지속 후보 {r.TEMP_EPISODES}개"] if r.TEMP_EPISODES else [])+
    (["CP 원본 검토"] if r.CP_REVIEW else [])+
    (["작업/감지 맥락"] if r.CAL_WORK+r.CAL_DETECT else [])+
    ([r.PLAN_RELATION] if r.PLAN_RELATION else [])),axis=1)
selection["ROLE"]=np.where(selection.REVIEW_REASON.ne(""),"검증 사례","같은 강종 대조·반대 사례 후보")
selection["NORMAL_LABEL"]="미확정"
quality_notes={}
for _,r in quality_review.iterrows():
    quality_notes[int(r["LOT"])]=f"OP-Z3 초기 NaN {int(r['NaN 개수'])}개; TC·OP 함께 유효한 1분창 {int(r['TC·OP 함께 유효한 1분창[개]'])}개"
selection["QUALITY_REVIEW"]=selection.LOT.map(quality_notes).fillna("공통 시각 누락·센서별 90% 미만 분을 확인")
table(selection)
note("**좁혀진 검증 범위:** 초기 승온은 가열 TC-Z3~6와 OP, 종료 전환은 가열 TC/OP와 CP, CP 후보는 TC-Z4·OP-Z4·CP-Z4M·냉각, 반복 파형은 TC-Z9·OUT1·OUT2, 감지는 실제 작업 시각과 대조한다. 온도 지속 후보는 전수 원본으로 확인한다. 240044·240061 OP-Z3 초기 추가 NaN은 품질 검토 대상이며 고장 후보로 자동 등록하지 않는다. SPHC·SPCE의 상대 비교, 다른 강종의 대조 부족, 5월 용접 전력 0을 각각 독립 가설로 03에 전달한다.")
note("**전체 공정 그림의 검토 위치 전수:** 색·알파벳·LOT·센서·경과 분·실제 시각·검토 이유·다음 검증을 함께 전달한다. 대표 표시 외의 온도·CP 후보는 7절 전수 표를 함께 사용한다.")
table(pd.DataFrame(page_reviews))
table(pd.DataFrame(figure_checks))
note(f"**시각화 확인:** {len(figure_checks)}개 그림에서 글자 겹침·잘림·범례 위치 검사를 통과했다.")